# SOLETE 5-Minute — Preprocessing and Feature Engineering

This notebook prepares the selected **5-minute SOLETE dataset** for the
forecasting experiments.

The 5-minute resolution was selected during the cross-resolution analysis
because it provides a useful balance between:

- temporal detail,
- data quality,
- computational efficiency,
- manageable sequence lengths,
- and preservation of the physical relationships observed at 1-minute
  resolution.

## Objectives

The preprocessing and feature-engineering workflow will address:

1. preservation of the original source data;
2. identified data-quality issues;
3. circular variables such as wind direction;
4. verification of uncertain variable encodings such as humidity;
5. realistic feature availability to prevent information leakage;
6. Gregorian calendar/time features;
7. astronomical and solar-position features;
8. lagged and historical forecasting features;
9. later integration of Vedic-calendar features.

## Methodological Principle

The original SOLETE measurements will not be silently overwritten.

Where preprocessing decisions are required, the notebook will:

- preserve the raw values,
- create explicit transformed or cleaned features,
- document the reason for each transformation,
- and keep preprocessing reproducible for later modelling.

Feature engineering will also distinguish between:

- information available at the forecast origin,
- historical/lagged measurements,
- deterministic future information,
- and future measurements that would not realistically be known at
  prediction time.

This distinction is essential for preventing information leakage.

In [1]:
# ============================================================
# CELL 1 — PROJECT SETUP AND LOAD SELECTED 5-MINUTE DATASET
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------
#
# This notebook is stored inside:
#
#     Project/notebooks/
#
# The following setup also works if Jupyter happens to be
# launched directly from the project root.

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# MAKE PROJECT MODULES IMPORTABLE
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


# Import our reusable SOLETE HDF5 loader.
from src.data_loader import load_solete


# ------------------------------------------------------------
# SELECTED DATASET
# ------------------------------------------------------------

dataset_path = (
    PROJECT_ROOT
    / "solete_dataset"
    / "SOLETE_Pombo_5min.h5"
)


# Verify that the expected dataset exists before loading it.
assert dataset_path.exists(), (
    f"Dataset not found: {dataset_path}"
)


# ------------------------------------------------------------
# LOAD ORIGINAL SOURCE DATA
# ------------------------------------------------------------
#
# df_raw is our reference copy.
#
# We will NOT intentionally modify this DataFrame during
# preprocessing or feature engineering.

df_raw = load_solete(
    dataset_path
)


# ------------------------------------------------------------
# CREATE A SEPARATE WORKING COPY
# ------------------------------------------------------------
#
# All transformations in this notebook will be performed
# on df_work.
#
# Keeping df_raw separate means we can always compare a
# transformed value with the original SOLETE measurement.

df_work = df_raw.copy()


# ------------------------------------------------------------
# BASIC VERIFICATION
# ------------------------------------------------------------

print("\nDataset shape:")
print(df_raw.shape)

print("\nDate range:")
print(
    df_raw.index.min(),
    "to",
    df_raw.index.max()
)

print("\nChronological order:")
print(
    df_raw.index.is_monotonic_increasing
)

print("\nDuplicate timestamps:")
print(
    df_raw.index.duplicated().sum()
)

print("\nTotal missing values:")
print(
    df_raw.isna().sum().sum()
)

print("\nWorking copy created:")
print(
    df_work is not df_raw
)

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

Dataset shape:
(131617, 11)

Date range:
2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00

Chronological order:
True

Duplicate timestamps:
0

Total missing values:
0

Working copy created:
True


In [2]:
# ============================================================
# CELL 2 — PREPROCESSING ISSUE REGISTER
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Before modifying or engineering any features, we create a
# compact register of the data-quality and representation
# issues identified during EDA.
#
# This cell does NOT modify df_raw or df_work.
#
# It simply re-verifies the important conditions that will
# influence our preprocessing decisions.


# ------------------------------------------------------------
# 1. IDENTIFY EXACT MIDNIGHT OBSERVATIONS
# ------------------------------------------------------------

midnight_mask = (
    (df_raw.index.hour == 0)
    &
    (df_raw.index.minute == 0)
)


# ------------------------------------------------------------
# 2. CALCULATE ISSUE COUNTS
# ------------------------------------------------------------

issue_register = pd.DataFrame(
    {
        "issue": [
            "Missing values",
            "Duplicate timestamps",
            "Wind direction > 360°",
            "Wind direction < 0°",
            "Pressure > 1100 mbar",
            "Pressure > 1100 mbar at midnight",
            "Gaia power > 12 kW",
            "Gaia power > 12 kW at midnight",
            "Negative wind speed",
            "Negative GHI",
            "Negative POA irradiance",
            "Negative solar power",
            "Negative Gaia power",
        ],

        "count": [
            df_raw.isna().sum().sum(),

            df_raw.index.duplicated().sum(),

            (
                df_raw["WIND_DIR[deg]"] > 360
            ).sum(),

            (
                df_raw["WIND_DIR[deg]"] < 0
            ).sum(),

            (
                df_raw["Pressure[mbar]"] > 1100
            ).sum(),

            (
                (df_raw["Pressure[mbar]"] > 1100)
                &
                midnight_mask
            ).sum(),

            (
                df_raw["P_Gaia[kW]"] > 12
            ).sum(),

            (
                (df_raw["P_Gaia[kW]"] > 12)
                &
                midnight_mask
            ).sum(),

            (
                df_raw["WIND_SPEED[m1s]"] < 0
            ).sum(),

            (
                df_raw["GHI[kW1m2]"] < 0
            ).sum(),

            (
                df_raw["POA Irr[kW1m2]"] < 0
            ).sum(),

            (
                df_raw["P_Solar[kW]"] < 0
            ).sum(),

            (
                df_raw["P_Gaia[kW]"] < 0
            ).sum(),
        ],
    }
)


# ------------------------------------------------------------
# 3. ADD PERCENTAGE OF COMPLETE DATASET
# ------------------------------------------------------------

issue_register["percentage"] = (
    issue_register["count"]
    / len(df_raw)
    * 100
)


# ------------------------------------------------------------
# 4. DISPLAY RESULTS
# ------------------------------------------------------------

issue_register

,issue,count,percentage
0,Missing values,0,0.000000
1,Duplicate timestamps,0,0.000000
2,Wind direction > 360°,103,0.078257
3,Wind direction < 0°,0,0.000000
4,Pressure > 1100 mbar,456,0.346460
5,Pressure > 1100 mbar at midnight,456,0.346460
6,Gaia power > 12 kW,51,0.038749
7,Gaia power > 12 kW at midnight,35,0.026592
8,Negative wind speed,0,0.000000
9,Negative GHI,0,0.000000


In [3]:
# ============================================================
# CELL 3 — FLAG CONFIRMED PRESSURE ANOMALIES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# EDA established that all 456 pressure observations above
# 1100 mbar occur exactly at 00:00 UTC and behave like
# systematic daily-boundary artifacts.
#
# We do NOT overwrite the original:
#
#     Pressure[mbar]
#
# Instead, we create:
#
#     pressure_anomaly_flag
#
#         1 -> confirmed suspicious pressure observation
#         0 -> otherwise
#
# and:
#
#     Pressure_clean[mbar]
#
# which is initially identical to the original pressure
# except that confirmed anomalous observations are marked
# as NaN.
#
# We are NOT filling/interpolating them yet.
# That decision will be evaluated separately.


# ------------------------------------------------------------
# 1. DEFINE THE CONFIRMED PRESSURE-ANOMALY CONDITION
# ------------------------------------------------------------

pressure_anomaly_mask = (
    (df_work["Pressure[mbar]"] > 1100)
    &
    (df_work.index.hour == 0)
    &
    (df_work.index.minute == 0)
)


# ------------------------------------------------------------
# 2. CREATE AN EXPLICIT ANOMALY FLAG
# ------------------------------------------------------------

df_work["pressure_anomaly_flag"] = (
    pressure_anomaly_mask.astype("int8")
)


# ------------------------------------------------------------
# 3. CREATE A SEPARATE CLEANED-PRESSURE CANDIDATE
# ------------------------------------------------------------

df_work["Pressure_clean[mbar]"] = (
    df_work["Pressure[mbar]"].copy()
)

df_work.loc[
    pressure_anomaly_mask,
    "Pressure_clean[mbar]"
] = np.nan


# ------------------------------------------------------------
# 4. VERIFY THE TRANSFORMATION
# ------------------------------------------------------------

print("Original pressure anomalies:")
print(
    (df_work["Pressure[mbar]"] > 1100).sum()
)

print("\nPressure anomaly flags:")
print(
    df_work["pressure_anomaly_flag"].sum()
)

print("\nMissing values introduced in cleaned pressure:")
print(
    df_work["Pressure_clean[mbar]"].isna().sum()
)

print("\nOriginal pressure column missing values:")
print(
    df_work["Pressure[mbar]"].isna().sum()
)


# ------------------------------------------------------------
# 5. VERIFY THAT df_raw WAS NOT MODIFIED
# ------------------------------------------------------------

print("\nMissing pressure values in df_raw:")
print(
    df_raw["Pressure[mbar]"].isna().sum()
)


# ------------------------------------------------------------
# 6. DISPLAY A FEW FLAGGED OBSERVATIONS
# ------------------------------------------------------------

pressure_flag_examples = df_work.loc[
    pressure_anomaly_mask,
    [
        "Pressure[mbar]",
        "Pressure_clean[mbar]",
        "pressure_anomaly_flag"
    ]
].head(10)

print("\nExample flagged pressure observations:")

pressure_flag_examples

Original pressure anomalies:
456

Pressure anomaly flags:
456

Missing values introduced in cleaned pressure:
456

Original pressure column missing values:
0

Missing pressure values in df_raw:
0

Example flagged pressure observations:


,Pressure[mbar],Pressure_clean[mbar],pressure_anomaly_flag
Timestamp,,,
2018-06-02 00:00:00+00:00,2013.659998,NaN,1
2018-06-03 00:00:00+00:00,2013.420020,NaN,1
2018-06-04 00:00:00+00:00,2011.500000,NaN,1
2018-06-05 00:00:00+00:00,2007.450345,NaN,1
2018-06-06 00:00:00+00:00,2015.010331,NaN,1
2018-06-07 00:00:00+00:00,2017.840002,NaN,1
2018-06-08 00:00:00+00:00,2014.500000,NaN,1
2018-06-09 00:00:00+00:00,2013.599976,NaN,1
2018-06-10 00:00:00+00:00,2011.733004,NaN,1


In [4]:
# ============================================================
# CELL 4 — COMPARE PRESSURE-CLEANING STRATEGIES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We have 456 confirmed pressure artifacts at 00:00 UTC.
#
# We now compare two possible preprocessing strategies:
#
#   A. Subtract 1000 mbar from the anomalous value
#
#   B. Replace the anomalous value with the immediately
#      previous valid observation (23:55)
#
# Both approaches avoid using future measurements.
#
# For DIAGNOSTIC purposes only, we also calculate a centered
# local reference:
#
#       mean(23:55, 00:05)
#
# IMPORTANT:
#
# The centered reference uses the future 00:05 observation,
# so it will NOT be used as a forecasting input.
#
# It is only used here to judge which causal cleaning
# strategy better follows the local pressure trajectory.


# ============================================================
# 1. CREATE CANDIDATE A — SUBTRACT 1000 MBAR
# ============================================================

df_work["Pressure_minus1000[mbar]"] = (
    df_work["Pressure[mbar]"].copy()
)

df_work.loc[
    pressure_anomaly_mask,
    "Pressure_minus1000[mbar]"
] = (
    df_work.loc[
        pressure_anomaly_mask,
        "Pressure[mbar]"
    ]
    - 1000
)


# ============================================================
# 2. CREATE CANDIDATE B — PREVIOUS VALID VALUE
# ============================================================

df_work["Pressure_ffill[mbar]"] = (
    df_work["Pressure_clean[mbar]"]
    .ffill()
)


# ============================================================
# 3. CREATE CENTERED LOCAL REFERENCE
# ============================================================
#
# This reference is ONLY for diagnostic comparison.
#
# Because observations are spaced exactly 5 minutes apart,
# the average of the previous and next observations gives a
# simple estimate of the local pressure trajectory.

pressure_neighbor_reference = (
    df_work["Pressure[mbar]"].shift(1)
    +
    df_work["Pressure[mbar]"].shift(-1)
) / 2


# ============================================================
# 4. COMPARE BOTH CANDIDATES AT ANOMALOUS TIMESTAMPS
# ============================================================

pressure_strategy_comparison = pd.DataFrame(
    {
        "original_anomalous":
            df_work.loc[
                pressure_anomaly_mask,
                "Pressure[mbar]"
            ],

        "subtract_1000":
            df_work.loc[
                pressure_anomaly_mask,
                "Pressure_minus1000[mbar]"
            ],

        "forward_fill":
            df_work.loc[
                pressure_anomaly_mask,
                "Pressure_ffill[mbar]"
            ],

        "neighbor_reference":
            pressure_neighbor_reference.loc[
                pressure_anomaly_mask
            ],
    }
)


# ------------------------------------------------------------
# Calculate absolute error relative to the diagnostic
# neighbor reference.
# ------------------------------------------------------------

pressure_strategy_comparison[
    "subtract_1000_error"
] = (
    pressure_strategy_comparison[
        "subtract_1000"
    ]
    -
    pressure_strategy_comparison[
        "neighbor_reference"
    ]
).abs()


pressure_strategy_comparison[
    "forward_fill_error"
] = (
    pressure_strategy_comparison[
        "forward_fill"
    ]
    -
    pressure_strategy_comparison[
        "neighbor_reference"
    ]
).abs()


# ============================================================
# 5. DISPLAY SUMMARY
# ============================================================

print(
    "Mean absolute deviation from centered "
    "neighbor reference:"
)

print(
    "\nSubtract-1000 strategy:"
)

print(
    pressure_strategy_comparison[
        "subtract_1000_error"
    ].mean()
)

print(
    "\nForward-fill strategy:"
)

print(
    pressure_strategy_comparison[
        "forward_fill_error"
    ].mean()
)


print(
    "\nMedian absolute deviation from centered "
    "neighbor reference:"
)

print(
    "\nSubtract-1000 strategy:"
)

print(
    pressure_strategy_comparison[
        "subtract_1000_error"
    ].median()
)

print(
    "\nForward-fill strategy:"
)

print(
    pressure_strategy_comparison[
        "forward_fill_error"
    ].median()
)


# ============================================================
# 6. SHOW EXAMPLE CORRECTIONS
# ============================================================

print("\nExample pressure-cleaning comparison:")

pressure_strategy_comparison.head(10)

Mean absolute deviation from centered neighbor reference:

Subtract-1000 strategy:
0.11183574475736373

Forward-fill strategy:
0.1146580871225296

Median absolute deviation from centered neighbor reference:

Subtract-1000 strategy:
0.0302482604981833

Forward-fill strategy:
0.022753295898155557

Example pressure-cleaning comparison:


,original_anomalous,subtract_1000,forward_fill,neighbor_reference,subtract_1000_error,forward_fill_error
Timestamp,,,,,,
2018-06-02 00:00:00+00:00,2013.659998,1013.659998,1013.362661,1013.514495,0.145503,0.151834
2018-06-03 00:00:00+00:00,2013.420020,1013.420020,1012.958009,1013.212175,0.207844,0.254166
2018-06-04 00:00:00+00:00,2011.500000,1011.500000,1011.165331,1011.327500,0.172500,0.162169
2018-06-05 00:00:00+00:00,2007.450345,1007.450345,1007.204684,1007.367172,0.083174,0.162488
2018-06-06 00:00:00+00:00,2015.010331,1015.010331,1014.626981,1014.828320,0.182010,0.201339
2018-06-07 00:00:00+00:00,2017.840002,1017.840002,1017.396999,1017.633333,0.206669,0.236334
2018-06-08 00:00:00+00:00,2014.500000,1014.500000,1014.045005,1014.242510,0.257490,0.197505
2018-06-09 00:00:00+00:00,2013.599976,1013.599976,1013.191989,1013.385984,0.213991,0.193996
2018-06-10 00:00:00+00:00,2011.733004,1011.733004,1011.362004,1011.534507,0.198497,0.172503


### Pressure Preprocessing Decision

The 456 confirmed pressure anomalies occur exactly at the daily
00:00 UTC boundary and are approximately 1000 mbar above the local
pressure trajectory.

Two causal correction strategies were evaluated:

1. subtract 1000 mbar from the anomalous observation;
2. replace the anomalous observation with the previous valid
   23:55 pressure value.

For diagnostic evaluation only, both strategies were compared with the
centered mean of the neighbouring 23:55 and 00:05 observations.

Results:

- subtract-1000 mean absolute deviation: approximately **0.112 mbar**
- forward-fill mean absolute deviation: approximately **0.115 mbar**
- subtract-1000 median absolute deviation: approximately **0.030 mbar**
- forward-fill median absolute deviation: approximately **0.023 mbar**

Both strategies reproduce the local pressure trajectory very closely.

The **subtract-1000 strategy is selected as the primary preprocessing
method** because the anomaly itself is a systematic approximately
+1000 mbar offset. This correction preserves information from the
affected timestamp while avoiding the use of future observations.

The original `Pressure[mbar]` measurement remains unchanged, and
`pressure_anomaly_flag` is retained so that corrected observations can
always be identified.

Forward-fill remains a documented alternative that may be tested in a
sensitivity analysis if required.

In [5]:
# ============================================================
# CELL 5 — FINALIZE CLEANED PRESSURE FEATURE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Based on Cell 4, the subtract-1000 strategy is selected
# as the primary correction for the 456 confirmed pressure
# artifacts.
#
# IMPORTANT:
#
# We do NOT modify:
#
#     Pressure[mbar]
#
# Instead, the final modelling candidate is stored in:
#
#     Pressure_clean[mbar]
#
# The anomaly flag is also retained.


# ------------------------------------------------------------
# 1. RECREATE CLEANED PRESSURE FROM THE ORIGINAL VALUES
# ------------------------------------------------------------

df_work["Pressure_clean[mbar]"] = (
    df_work["Pressure[mbar]"].copy()
)


# ------------------------------------------------------------
# 2. APPLY THE SELECTED CORRECTION ONLY TO CONFIRMED
#    ANOMALOUS OBSERVATIONS
# ------------------------------------------------------------

df_work.loc[
    pressure_anomaly_mask,
    "Pressure_clean[mbar]"
] = (
    df_work.loc[
        pressure_anomaly_mask,
        "Pressure[mbar]"
    ]
    - 1000
)


# ------------------------------------------------------------
# 3. VERIFY THE FINAL CLEANED FEATURE
# ------------------------------------------------------------

print("Original pressure range:")
print(
    df_work["Pressure[mbar]"].min(),
    "to",
    df_work["Pressure[mbar]"].max()
)

print("\nCleaned pressure range:")
print(
    df_work["Pressure_clean[mbar]"].min(),
    "to",
    df_work["Pressure_clean[mbar]"].max()
)

print("\nMissing values in cleaned pressure:")
print(
    df_work["Pressure_clean[mbar]"].isna().sum()
)

print("\nPressure values above 1100 mbar after correction:")
print(
    (
        df_work["Pressure_clean[mbar]"] > 1100
    ).sum()
)

print("\nNumber of corrected observations:")
print(
    df_work["pressure_anomaly_flag"].sum()
)


# ------------------------------------------------------------
# 4. CONFIRM RAW PRESSURE IS STILL UNCHANGED
# ------------------------------------------------------------

print("\nRaw and working original pressure identical?")
print(
    df_raw["Pressure[mbar]"].equals(
        df_work["Pressure[mbar]"]
    )
)


# ------------------------------------------------------------
# 5. DISPLAY EXAMPLE CORRECTIONS
# ------------------------------------------------------------

pressure_final_examples = df_work.loc[
    pressure_anomaly_mask,
    [
        "Pressure[mbar]",
        "Pressure_clean[mbar]",
        "pressure_anomaly_flag"
    ]
].head(10)

print("\nExample final pressure corrections:")

pressure_final_examples

Original pressure range:
974.2629986572266 to 2034.5196618652349

Cleaned pressure range:
974.2629986572266 to 1042.3197102864617

Missing values in cleaned pressure:
0

Pressure values above 1100 mbar after correction:
0

Number of corrected observations:
456

Raw and working original pressure identical?
True

Example final pressure corrections:


,Pressure[mbar],Pressure_clean[mbar],pressure_anomaly_flag
Timestamp,,,
2018-06-02 00:00:00+00:00,2013.659998,1013.659998,1
2018-06-03 00:00:00+00:00,2013.420020,1013.420020,1
2018-06-04 00:00:00+00:00,2011.500000,1011.500000,1
2018-06-05 00:00:00+00:00,2007.450345,1007.450345,1
2018-06-06 00:00:00+00:00,2015.010331,1015.010331,1
2018-06-07 00:00:00+00:00,2017.840002,1017.840002,1
2018-06-08 00:00:00+00:00,2014.500000,1014.500000,1
2018-06-09 00:00:00+00:00,2013.599976,1013.599976,1
2018-06-10 00:00:00+00:00,2011.733004,1011.733004,1


In [6]:
# ============================================================
# CELL 6 — CREATE CIRCULAR WIND-DIRECTION FEATURES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Wind direction is a CIRCULAR variable.
#
# Examples:
#
#     1° and 359°
#
# are physically very close even though their ordinary
# numerical difference is 358°.
#
# Therefore, raw wind direction should not be treated as a
# normal linear numerical feature in a forecasting model.
#
# EDA also found 103 raw wind-direction observations above
# 360 degrees.
#
# We preserve the original:
#
#     WIND_DIR[deg]
#
# and create:
#
#     WIND_DIR_wrapped[deg]
#     wind_dir_sin
#     wind_dir_cos
#     wind_dir_wrap_flag
#
# The original SOLETE measurements remain unchanged.


# ------------------------------------------------------------
# 1. FLAG RAW VALUES OUTSIDE THE CONVENTIONAL RANGE
# ------------------------------------------------------------
#
# The 5-minute data contain no negative wind directions,
# but we define the condition generally.

wind_direction_wrap_mask = (
    (df_work["WIND_DIR[deg]"] < 0)
    |
    (df_work["WIND_DIR[deg]"] >= 360)
)


df_work["wind_dir_wrap_flag"] = (
    wind_direction_wrap_mask.astype("int8")
)


# ------------------------------------------------------------
# 2. WRAP WIND DIRECTION INTO [0°, 360°)
# ------------------------------------------------------------
#
# Examples:
#
#     360° ->   0°
#     450° ->  90°
#     560° -> 200°
#
# This creates a new modelling-oriented representation.
# The raw WIND_DIR[deg] column is NOT overwritten.

df_work["WIND_DIR_wrapped[deg]"] = (
    df_work["WIND_DIR[deg]"] % 360
)


# ------------------------------------------------------------
# 3. CONVERT WRAPPED ANGLES TO RADIANS
# ------------------------------------------------------------

wind_direction_radians = np.deg2rad(
    df_work["WIND_DIR_wrapped[deg]"]
)


# ------------------------------------------------------------
# 4. CREATE SINE AND COSINE FEATURES
# ------------------------------------------------------------
#
# Together, sine and cosine preserve the circular geometry
# of wind direction.
#
# For example:
#
#     0°   -> sin =  0, cos =  1
#     90°  -> sin =  1, cos =  0
#     180° -> sin =  0, cos = -1
#     270° -> sin = -1, cos =  0

df_work["wind_dir_sin"] = np.sin(
    wind_direction_radians
)

df_work["wind_dir_cos"] = np.cos(
    wind_direction_radians
)


# ------------------------------------------------------------
# 5. VERIFY THE TRANSFORMATION
# ------------------------------------------------------------

print("Raw wind-direction range:")
print(
    df_work["WIND_DIR[deg]"].min(),
    "to",
    df_work["WIND_DIR[deg]"].max()
)

print("\nWrapped wind-direction range:")
print(
    df_work["WIND_DIR_wrapped[deg]"].min(),
    "to",
    df_work["WIND_DIR_wrapped[deg]"].max()
)

print("\nNumber of raw values requiring wrapping:")
print(
    df_work["wind_dir_wrap_flag"].sum()
)

print("\nSine feature range:")
print(
    df_work["wind_dir_sin"].min(),
    "to",
    df_work["wind_dir_sin"].max()
)

print("\nCosine feature range:")
print(
    df_work["wind_dir_cos"].min(),
    "to",
    df_work["wind_dir_cos"].max()
)


# ------------------------------------------------------------
# 6. CHECK CIRCULAR REPRESENTATION
# ------------------------------------------------------------
#
# For every angle:
#
#     sin²(theta) + cos²(theta) = 1
#
# Small deviations may occur only because of floating-point
# precision.

circular_norm = (
    df_work["wind_dir_sin"] ** 2
    +
    df_work["wind_dir_cos"] ** 2
)

print("\nMaximum deviation from sin² + cos² = 1:")
print(
    np.abs(
        circular_norm - 1
    ).max()
)


# ------------------------------------------------------------
# 7. SHOW EXAMPLES OF WRAPPED VALUES
# ------------------------------------------------------------

wind_direction_examples = df_work.loc[
    wind_direction_wrap_mask,
    [
        "WIND_DIR[deg]",
        "WIND_DIR_wrapped[deg]",
        "wind_dir_sin",
        "wind_dir_cos",
        "wind_dir_wrap_flag"
    ]
].head(10)

print("\nExamples of wrapped wind directions:")

wind_direction_examples

Raw wind-direction range:
0.0 to 680.0

Wrapped wind-direction range:
0.0 to 356.6387959866221

Number of raw values requiring wrapping:
103

Sine feature range:
-1.0 to 1.0

Cosine feature range:
-0.999999998307681 to 1.0

Maximum deviation from sin² + cos² = 1:
2.220446049250313e-16

Examples of wrapped wind directions:


,WIND_DIR[deg],WIND_DIR_wrapped[deg],wind_dir_sin,wind_dir_cos,wind_dir_wrap_flag
Timestamp,,,,,
2018-06-04 00:00:00+00:00,531.528428,171.528428,0.147319,-0.989089,1
2018-06-12 00:00:00+00:00,565.193980,205.193980,-0.425684,-0.904872,1
2018-06-13 00:00:00+00:00,489.896321,129.896321,0.767206,-0.641400,1
2018-06-14 00:00:00+00:00,470.468227,110.468227,0.936866,-0.349688,1
2018-06-18 00:00:00+00:00,457.521739,97.521739,0.991395,-0.130902,1
2018-06-21 00:00:00+00:00,449.832776,89.832776,0.999996,0.002919,1
2018-06-22 00:00:00+00:00,538.444816,178.444816,0.027140,-0.999632,1
2018-06-24 00:00:00+00:00,468.147157,108.147157,0.950260,-0.311459,1
2018-06-29 00:00:00+00:00,528.739130,168.739130,0.195276,-0.980748,1


### Wind-Direction Feature Decision

Wind direction is retained in its original form for traceability, but the
raw degree value will not be used directly as the default modelling feature.

The preprocessing pipeline creates:

- `WIND_DIR_wrapped[deg]` — diagnostic angle mapped into `[0°, 360°)`;
- `wind_dir_sin` — circular sine component;
- `wind_dir_cos` — circular cosine component;
- `wind_dir_wrap_flag` — diagnostic indicator for source values that required
  modulo-360 wrapping.

The primary modelling representation will use:

- `wind_dir_sin`
- `wind_dir_cos`

This avoids the artificial discontinuity between angles such as `359°`
and `1°`.

`WIND_DIR_wrapped[deg]` and `wind_dir_wrap_flag` are retained for
interpretation and diagnostics rather than included by default in the
forecasting feature set.

In [7]:
# ============================================================
# CELL 7 — INVESTIGATE HUMIDITY ENCODING AND BOUNDARY BEHAVIOUR
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The SOLETE column is labelled:
#
#     HUMIDITY[%]
#
# However, the observed values do not resemble conventional
# relative-humidity percentage points on a 0–100 scale.
#
# Most observations are around values such as:
#
#     0.7
#     0.8
#     0.9
#
# while some values exceed 1.
#
# Before applying any conversion, we investigate whether
# values above 1 are:
#
#   - distributed throughout the dataset,
#   - concentrated at 00:00 UTC,
#   - or associated with another processing artifact.
#
# No humidity values are modified in this cell.


# ------------------------------------------------------------
# 1. BASIC HUMIDITY RANGE
# ------------------------------------------------------------

humidity = df_raw["HUMIDITY[%]"]

print("Humidity range:")
print(
    humidity.min(),
    "to",
    humidity.max()
)

print("\nHumidity summary:")
print(
    humidity.describe()
)


# ------------------------------------------------------------
# 2. COUNT VALUES ABOVE COMMON FRACTIONAL LIMITS
# ------------------------------------------------------------
#
# If humidity were stored as a fraction, conventional
# relative humidity would usually lie approximately within:
#
#     0 <= humidity <= 1
#
# We therefore inspect values above 1 without yet assuming
# that this interpretation is correct.

humidity_above_1 = (
    humidity > 1
)

print("\nHumidity observations > 1:")
print(
    humidity_above_1.sum()
)

print("\nPercentage > 1:")
print(
    f"{humidity_above_1.mean() * 100:.4f}%"
)


# ------------------------------------------------------------
# 3. CHECK WHETHER VALUES > 1 OCCUR AT MIDNIGHT
# ------------------------------------------------------------

humidity_above_1_at_midnight = (
    humidity_above_1
    &
    midnight_mask
)

print("\nHumidity > 1 occurring at 00:00 UTC:")
print(
    humidity_above_1_at_midnight.sum()
)


# ------------------------------------------------------------
# 4. DISTRIBUTION BY HOUR OF DAY
# ------------------------------------------------------------

humidity_high_by_hour = (
    pd.DataFrame({
        "humidity_above_1":
            humidity_above_1.astype(int)
    })
    .assign(
        hour=df_raw.index.hour
    )
    .groupby("hour")
    ["humidity_above_1"]
    .sum()
)

print("\nNumber of humidity > 1 observations by UTC hour:")
print(
    humidity_high_by_hour
)


# ------------------------------------------------------------
# 5. SHOW THE HIGHEST HUMIDITY OBSERVATIONS
# ------------------------------------------------------------

highest_humidity = (
    df_raw[
        ["HUMIDITY[%]"]
    ]
    .sort_values(
        "HUMIDITY[%]",
        ascending=False
    )
    .head(20)
)

print("\nHighest humidity observations:")
highest_humidity

Humidity range:
0.0 to 2.0

Humidity summary:
count    131617.000000
mean          0.769125
std           0.158512
min           0.000000
25%           0.700000
50%           0.800000
75%           0.900000
max           2.000000
Name: HUMIDITY[%], dtype: float64

Humidity observations > 1:
164

Percentage > 1:
0.1246%

Humidity > 1 occurring at 00:00 UTC:
164

Number of humidity > 1 observations by UTC hour:
hour
0     164
1       0
2       0
3       0
4       0
5       0
6       0
7       0
8       0
9       0
10      0
11      0
12      0
13      0
14      0
15      0
16      0
17      0
18      0
19      0
20      0
21      0
22      0
23      0
Name: humidity_above_1, dtype: int64

Highest humidity observations:


,HUMIDITY[%]
Timestamp,
2018-11-13 00:00:00+00:00,2.000000
2019-03-06 00:00:00+00:00,2.000000
2019-03-04 00:00:00+00:00,2.000000
2018-12-20 00:00:00+00:00,2.000000
2018-11-07 00:00:00+00:00,2.000000
2019-01-13 00:00:00+00:00,2.000000
2018-12-30 00:00:00+00:00,2.000000
2019-02-25 00:00:00+00:00,2.000000
2018-09-05 00:00:00+00:00,2.000000


In [8]:
# ============================================================
# CELL 8 — COMPARE HUMIDITY-CLEANING STRATEGIES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 7 showed that:
#
#   - 164 humidity observations are greater than 1;
#   - ALL 164 occur exactly at 00:00 UTC.
#
# This strongly suggests a systematic daily-boundary artifact.
#
# The suspicious values often appear approximately 1 unit
# above the normal humidity scale, so we test:
#
#   A. subtract 1.0 from the suspicious value
#
#   B. replace it with the immediately previous valid value
#      (23:55)
#
# For DIAGNOSTIC purposes only, both are compared against:
#
#       mean(23:55, 00:05)
#
# The centered reference uses a future observation and will
# therefore NOT be used as a forecasting feature.
#
# The original HUMIDITY[%] column is NOT modified.


# ============================================================
# 1. DEFINE CONFIRMED SUSPICIOUS HUMIDITY OBSERVATIONS
# ============================================================

humidity_anomaly_mask = (
    (df_work["HUMIDITY[%]"] > 1)
    &
    (df_work.index.hour == 0)
    &
    (df_work.index.minute == 0)
)


# ============================================================
# 2. CREATE ANOMALY FLAG
# ============================================================

df_work["humidity_anomaly_flag"] = (
    humidity_anomaly_mask.astype("int8")
)


# ============================================================
# 3. CANDIDATE A — SUBTRACT 1.0
# ============================================================

humidity_subtract_1 = (
    df_work["HUMIDITY[%]"].copy()
)

humidity_subtract_1.loc[
    humidity_anomaly_mask
] = (
    humidity_subtract_1.loc[
        humidity_anomaly_mask
    ]
    - 1.0
)


# ============================================================
# 4. CANDIDATE B — PREVIOUS VALID OBSERVATION
# ============================================================

humidity_without_anomalies = (
    df_work["HUMIDITY[%]"].copy()
)

humidity_without_anomalies.loc[
    humidity_anomaly_mask
] = np.nan

humidity_forward_fill = (
    humidity_without_anomalies.ffill()
)


# ============================================================
# 5. CENTERED LOCAL REFERENCE — DIAGNOSTIC ONLY
# ============================================================

humidity_neighbor_reference = (
    df_work["HUMIDITY[%]"].shift(1)
    +
    df_work["HUMIDITY[%]"].shift(-1)
) / 2


# ============================================================
# 6. BUILD STRATEGY COMPARISON
# ============================================================

humidity_strategy_comparison = pd.DataFrame(
    {
        "original_anomalous":
            df_work.loc[
                humidity_anomaly_mask,
                "HUMIDITY[%]"
            ],

        "subtract_1":
            humidity_subtract_1.loc[
                humidity_anomaly_mask
            ],

        "forward_fill":
            humidity_forward_fill.loc[
                humidity_anomaly_mask
            ],

        "neighbor_reference":
            humidity_neighbor_reference.loc[
                humidity_anomaly_mask
            ],
    }
)


# ------------------------------------------------------------
# Absolute deviation from diagnostic local reference
# ------------------------------------------------------------

humidity_strategy_comparison[
    "subtract_1_error"
] = (
    humidity_strategy_comparison["subtract_1"]
    -
    humidity_strategy_comparison["neighbor_reference"]
).abs()


humidity_strategy_comparison[
    "forward_fill_error"
] = (
    humidity_strategy_comparison["forward_fill"]
    -
    humidity_strategy_comparison["neighbor_reference"]
).abs()


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

print("Number of suspicious humidity observations:")
print(
    humidity_anomaly_mask.sum()
)


print(
    "\nMean absolute deviation from centered "
    "neighbor reference:"
)

print("\nSubtract-1 strategy:")
print(
    humidity_strategy_comparison[
        "subtract_1_error"
    ].mean()
)

print("\nForward-fill strategy:")
print(
    humidity_strategy_comparison[
        "forward_fill_error"
    ].mean()
)


print(
    "\nMedian absolute deviation from centered "
    "neighbor reference:"
)

print("\nSubtract-1 strategy:")
print(
    humidity_strategy_comparison[
        "subtract_1_error"
    ].median()
)

print("\nForward-fill strategy:")
print(
    humidity_strategy_comparison[
        "forward_fill_error"
    ].median()
)


print("\nRange after subtract-1 correction:")
print(
    humidity_subtract_1.loc[
        humidity_anomaly_mask
    ].min(),
    "to",
    humidity_subtract_1.loc[
        humidity_anomaly_mask
    ].max()
)


print("\nExample humidity-cleaning comparison:")

humidity_strategy_comparison.head(15)

Number of suspicious humidity observations:
164

Mean absolute deviation from centered neighbor reference:

Subtract-1 strategy:
0.15363437406531255

Forward-fill strategy:
0.002198170731707318

Median absolute deviation from centered neighbor reference:

Subtract-1 strategy:
0.17483333333333323

Forward-fill strategy:
0.0

Range after subtract-1 correction:
0.19999999999999996 to 1.0

Example humidity-cleaning comparison:


,original_anomalous,subtract_1,forward_fill,neighbor_reference,subtract_1_error,forward_fill_error
Timestamp,,,,,,
2018-06-02 00:00:00+00:00,1.800000,0.800000,0.900000,0.900000,0.100000,0.000000
2018-06-04 00:00:00+00:00,1.800000,0.800000,0.900000,0.900000,0.100000,0.000000
2018-06-09 00:00:00+00:00,1.600000,0.600000,0.778000,0.789000,0.189000,0.011000
2018-06-10 00:00:00+00:00,1.600000,0.600000,0.800000,0.811833,0.211833,0.011833
2018-06-12 00:00:00+00:00,1.600000,0.600000,0.800000,0.800000,0.200000,0.000000
2018-06-13 00:00:00+00:00,1.800000,0.800000,0.900000,0.900000,0.100000,0.000000
2018-06-14 00:00:00+00:00,1.501003,0.501003,0.758667,0.729333,0.228330,0.029333
2018-06-16 00:00:00+00:00,1.600000,0.600000,0.800000,0.800000,0.200000,0.000000
2018-06-18 00:00:00+00:00,1.600000,0.600000,0.800000,0.800000,0.200000,0.000000


### Humidity Preprocessing Decision

The 5-minute dataset contains 164 humidity observations greater than 1,
and all 164 occur exactly at 00:00 UTC.

Two causal correction strategies were evaluated:

1. subtract 1.0 from the suspicious midnight value;
2. replace the suspicious value with the immediately preceding valid
   23:55 observation.

For diagnostic purposes only, both strategies were compared with the
mean of the neighbouring 23:55 and 00:05 observations.

Results:

- subtract-1 mean absolute deviation: approximately **0.154**
- forward-fill mean absolute deviation: approximately **0.0022**
- subtract-1 median absolute deviation: approximately **0.175**
- forward-fill median absolute deviation: **0.0**

The forward-fill strategy is therefore strongly preferred.

The results do **not** support interpreting the anomalous observations
as a simple +1 offset. Instead, they appear to be isolated
daily-boundary corruptions whose numerical values should not be trusted.

The selected correction replaces only the 164 confirmed values greater
than 1 at 00:00 UTC with the immediately preceding valid observation.
This is a causal correction because no future measurement is used.

The original `HUMIDITY[%]` column remains unchanged, and an explicit
`humidity_anomaly_flag` is retained.

The physical/unit interpretation of the humidity scale remains separate
from this artifact correction. No multiplication by 100 or other unit
conversion is applied at this stage.

In [9]:
# ============================================================
# CELL 9 — FINALIZE CLEANED HUMIDITY FEATURE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 8 showed that forward-filling from the immediately
# preceding valid observation is much more consistent with
# the local humidity trajectory than subtracting 1.0.
#
# We therefore create:
#
#     Humidity_clean
#
# while retaining:
#
#     HUMIDITY[%]
#
# unchanged.
#
# Only the 164 confirmed suspicious observations are
# replaced.
#
# No unit conversion is performed here.


# ------------------------------------------------------------
# 1. CREATE CLEANED HUMIDITY FROM ORIGINAL VALUES
# ------------------------------------------------------------

df_work["Humidity_clean"] = (
    df_work["HUMIDITY[%]"].copy()
)


# ------------------------------------------------------------
# 2. TEMPORARILY MARK CONFIRMED ANOMALIES AS MISSING
# ------------------------------------------------------------

df_work.loc[
    humidity_anomaly_mask,
    "Humidity_clean"
] = np.nan


# ------------------------------------------------------------
# 3. FORWARD-FILL ONLY THE MISSING ANOMALOUS VALUES
# ------------------------------------------------------------
#
# Because the dataset is chronologically ordered at an exact
# 5-minute interval, this uses the immediately preceding
# observation (23:55 for each affected midnight record).
#
# This is causal: no future observation is used.

df_work["Humidity_clean"] = (
    df_work["Humidity_clean"].ffill()
)


# ------------------------------------------------------------
# 4. VERIFY THE RESULT
# ------------------------------------------------------------

print("Original humidity range:")
print(
    df_work["HUMIDITY[%]"].min(),
    "to",
    df_work["HUMIDITY[%]"].max()
)

print("\nCleaned humidity range:")
print(
    df_work["Humidity_clean"].min(),
    "to",
    df_work["Humidity_clean"].max()
)

print("\nMissing values in cleaned humidity:")
print(
    df_work["Humidity_clean"].isna().sum()
)

print("\nCleaned humidity observations > 1:")
print(
    (
        df_work["Humidity_clean"] > 1
    ).sum()
)

print("\nNumber of humidity anomaly flags:")
print(
    df_work["humidity_anomaly_flag"].sum()
)


# ------------------------------------------------------------
# 5. CONFIRM ORIGINAL SOURCE COLUMN IS UNCHANGED
# ------------------------------------------------------------

print("\nRaw and working original humidity identical?")
print(
    df_raw["HUMIDITY[%]"].equals(
        df_work["HUMIDITY[%]"]
    )
)


# ------------------------------------------------------------
# 6. DISPLAY EXAMPLE CORRECTIONS
# ------------------------------------------------------------

humidity_final_examples = df_work.loc[
    humidity_anomaly_mask,
    [
        "HUMIDITY[%]",
        "Humidity_clean",
        "humidity_anomaly_flag"
    ]
].head(15)

print("\nExample final humidity corrections:")

humidity_final_examples

Original humidity range:
0.0 to 2.0

Cleaned humidity range:
0.0 to 1.0

Missing values in cleaned humidity:
0

Cleaned humidity observations > 1:
0

Number of humidity anomaly flags:
164

Raw and working original humidity identical?
True

Example final humidity corrections:


,HUMIDITY[%],Humidity_clean,humidity_anomaly_flag
Timestamp,,,
2018-06-02 00:00:00+00:00,1.800000,0.900000,1
2018-06-04 00:00:00+00:00,1.800000,0.900000,1
2018-06-09 00:00:00+00:00,1.600000,0.778000,1
2018-06-10 00:00:00+00:00,1.600000,0.800000,1
2018-06-12 00:00:00+00:00,1.600000,0.800000,1
2018-06-13 00:00:00+00:00,1.800000,0.900000,1
2018-06-14 00:00:00+00:00,1.501003,0.758667,1
2018-06-16 00:00:00+00:00,1.600000,0.800000,1
2018-06-18 00:00:00+00:00,1.600000,0.800000,1


### Humidity Cleaning Outcome

The selected humidity correction has been applied successfully.

- The original `HUMIDITY[%]` measurement remains unchanged.
- Exactly 164 confirmed anomalous midnight observations are flagged.
- `Humidity_clean` contains no missing values.
- Its observed range is now 0 to 1.
- No multiplication by 100 or other unit conversion has been applied.

The cleaned 0–1 range is treated as the dataset's observed modelling
scale. Its exact physical/unit interpretation will remain explicitly
documented as uncertain until verified from the source documentation.

In [10]:
# ============================================================
# CELL 10 — INVESTIGATE EXTREME GAIA-POWER OBSERVATIONS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# P_Gaia[kW] may later be used as a forecasting target.
#
# Target values require especially conservative preprocessing:
# we should not remove or alter them merely because they exceed
# the nominal turbine rating.
#
# EDA showed:
#
#   - some values slightly above 12 kW occur outside midnight;
#   - much larger peaks are concentrated around 00:00 UTC.
#
# Therefore, this cell does NOT clean P_Gaia.
#
# Instead, we measure how extreme observations differ from
# their immediately adjacent 5-minute observations.


# ------------------------------------------------------------
# 1. CREATE USEFUL MASKS
# ------------------------------------------------------------

gaia = df_raw["P_Gaia[kW]"]

gaia_above_12_mask = (
    gaia > 12
)

gaia_above_12_midnight_mask = (
    gaia_above_12_mask
    &
    midnight_mask
)

gaia_above_12_non_midnight_mask = (
    gaia_above_12_mask
    &
    ~midnight_mask
)


# ------------------------------------------------------------
# 2. BASIC COUNTS
# ------------------------------------------------------------

print("P_Gaia observations > 12 kW:")
print(
    gaia_above_12_mask.sum()
)

print("\n> 12 kW at midnight:")
print(
    gaia_above_12_midnight_mask.sum()
)

print("\n> 12 kW outside midnight:")
print(
    gaia_above_12_non_midnight_mask.sum()
)

print("\nMaximum P_Gaia at midnight:")
print(
    gaia.loc[midnight_mask].max()
)

print("\nMaximum P_Gaia outside midnight:")
print(
    gaia.loc[~midnight_mask].max()
)


# ------------------------------------------------------------
# 3. CREATE LOCAL NEIGHBOUR REFERENCE
# ------------------------------------------------------------
#
# Diagnostic only:
#
#     mean(previous 5-minute value, next 5-minute value)
#
# This is NOT used as a forecasting feature or correction.

gaia_neighbor_reference = (
    gaia.shift(1)
    +
    gaia.shift(-1)
) / 2


# ------------------------------------------------------------
# 4. MEASURE LOCAL DEVIATION FOR >12 kW OBSERVATIONS
# ------------------------------------------------------------

gaia_high_comparison = pd.DataFrame(
    {
        "P_Gaia":
            gaia.loc[
                gaia_above_12_mask
            ],

        "previous":
            gaia.shift(1).loc[
                gaia_above_12_mask
            ],

        "next":
            gaia.shift(-1).loc[
                gaia_above_12_mask
            ],

        "neighbor_reference":
            gaia_neighbor_reference.loc[
                gaia_above_12_mask
            ],

        "is_midnight":
            midnight_mask[
                gaia_above_12_mask
            ],
    }
)


gaia_high_comparison[
    "deviation_from_neighbors"
] = (
    gaia_high_comparison["P_Gaia"]
    -
    gaia_high_comparison["neighbor_reference"]
).abs()


# ------------------------------------------------------------
# 5. COMPARE MIDNIGHT VS NON-MIDNIGHT HIGH-POWER VALUES
# ------------------------------------------------------------

print(
    "\nMedian local deviation for >12 kW MIDNIGHT values:"
)

print(
    gaia_high_comparison.loc[
        gaia_high_comparison["is_midnight"],
        "deviation_from_neighbors"
    ].median()
)


print(
    "\nMedian local deviation for >12 kW NON-MIDNIGHT values:"
)

print(
    gaia_high_comparison.loc[
        ~gaia_high_comparison["is_midnight"],
        "deviation_from_neighbors"
    ].median()
)


print(
    "\nMean local deviation for >12 kW MIDNIGHT values:"
)

print(
    gaia_high_comparison.loc[
        gaia_high_comparison["is_midnight"],
        "deviation_from_neighbors"
    ].mean()
)


print(
    "\nMean local deviation for >12 kW NON-MIDNIGHT values:"
)

print(
    gaia_high_comparison.loc[
        ~gaia_high_comparison["is_midnight"],
        "deviation_from_neighbors"
    ].mean()
)


# ------------------------------------------------------------
# 6. SHOW THE LARGEST GAIA VALUES
# ------------------------------------------------------------

print("\nLargest P_Gaia observations:")

gaia_high_comparison.sort_values(
    "P_Gaia",
    ascending=False
).head(15)

P_Gaia observations > 12 kW:
51

> 12 kW at midnight:
35

> 12 kW outside midnight:
16

Maximum P_Gaia at midnight:
23.33053846153846

Maximum P_Gaia outside midnight:
12.509403333333333

Median local deviation for >12 kW MIDNIGHT values:
9.273171443701226

Median local deviation for >12 kW NON-MIDNIGHT values:
0.20429166666666898

Mean local deviation for >12 kW MIDNIGHT values:
9.195104887880237

Mean local deviation for >12 kW NON-MIDNIGHT values:
0.5016469791666671

Largest P_Gaia observations:


,P_Gaia,previous,next,neighbor_reference,is_midnight,deviation_from_neighbors
Timestamp,,,,,,
2018-12-08 00:00:00+00:00,23.330538,10.959943,11.373940,11.166942,True,12.163597
2019-03-16 00:00:00+00:00,23.116615,10.263037,11.803360,11.033198,True,12.083417
2018-09-24 00:00:00+00:00,22.876445,10.508380,10.518813,10.513597,True,12.362848
2019-01-01 00:00:00+00:00,22.821806,10.403780,11.457090,10.930435,True,11.891371
2019-02-05 00:00:00+00:00,22.194719,9.204190,9.710230,9.457210,True,12.737509
2018-09-28 00:00:00+00:00,22.148933,11.206147,10.724177,10.965162,True,11.183771
2018-08-12 00:00:00+00:00,21.930482,10.978423,10.614727,10.796575,True,11.133907
2019-01-15 00:00:00+00:00,21.750007,10.521023,11.516587,11.018805,True,10.731202
2018-12-10 00:00:00+00:00,21.507853,10.251413,10.601850,10.426632,True,11.081221


### Preliminary Gaia-Power Anomaly Finding

High Gaia-power observations show a strong distinction between midnight
and non-midnight behaviour.

For observations above 12 kW:

- midnight values have a median local deviation of approximately **9.27 kW**;
- non-midnight values have a median local deviation of approximately **0.20 kW**.

The largest midnight observations reach approximately 20–23 kW while
their immediately adjacent observations are often only about 8–11 kW.

In contrast, non-midnight values slightly above 12 kW are generally much
more consistent with their local temporal neighbourhood.

Therefore:

- `P_Gaia > 12 kW` must **not** be treated as a universal invalid-value rule;
- extreme midnight observations are strong anomaly candidates;
- no Gaia-power values are modified yet because `P_Gaia[kW]` may later
  serve as a forecasting target.

Before defining a target-cleaning strategy, the complete midnight
distribution will be examined to determine whether the artifact is
limited to values above 12 kW or affects additional midnight observations.

In [11]:
# ============================================================
# CELL 11 — EXAMINE ALL MIDNIGHT GAIA DISCONTINUITIES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 10 established that midnight P_Gaia values above
# 12 kW are extremely inconsistent with their neighbouring
# observations.
#
# But using:
#
#     P_Gaia > 12 kW
#
# as the anomaly definition could miss corrupted midnight
# values whose absolute power happens to remain below 12 kW.
#
# Therefore, this cell examines ALL midnight Gaia values,
# regardless of their absolute magnitude.
#
# No target values are changed.


# ------------------------------------------------------------
# 1. CALCULATE LOCAL DEVIATION FOR EVERY INTERIOR TIMESTAMP
# ------------------------------------------------------------

gaia_local_deviation = (
    gaia
    -
    gaia_neighbor_reference
).abs()


# ------------------------------------------------------------
# 2. COMPARE MIDNIGHT AND NON-MIDNIGHT DISTRIBUTIONS
# ------------------------------------------------------------

midnight_deviation = (
    gaia_local_deviation.loc[midnight_mask]
    .dropna()
)

non_midnight_deviation = (
    gaia_local_deviation.loc[~midnight_mask]
    .dropna()
)


print("Number of midnight observations:")
print(
    len(midnight_deviation)
)


print("\nMidnight local-deviation summary:")
print(
    midnight_deviation.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


print("\nNon-midnight local-deviation summary:")
print(
    non_midnight_deviation.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 3. COUNT LARGE MIDNIGHT DISCONTINUITIES
# ------------------------------------------------------------
#
# These thresholds are diagnostic only.
#
# We are NOT defining anomalies from them yet.

for threshold in [1, 2, 3, 5, 8, 10]:
    count = (
        midnight_deviation > threshold
    ).sum()

    print(
        f"\nMidnight observations with local "
        f"deviation > {threshold} kW:"
    )
    print(count)


# ------------------------------------------------------------
# 4. SHOW THE LARGEST MIDNIGHT DISCONTINUITIES
# ------------------------------------------------------------

gaia_midnight_diagnostics = pd.DataFrame(
    {
        "P_Gaia":
            gaia.loc[midnight_mask],

        "previous":
            gaia.shift(1).loc[midnight_mask],

        "next":
            gaia.shift(-1).loc[midnight_mask],

        "neighbor_reference":
            gaia_neighbor_reference.loc[
                midnight_mask
            ],

        "local_deviation":
            gaia_local_deviation.loc[
                midnight_mask
            ],
    }
)


print("\nLargest midnight Gaia discontinuities:")

gaia_midnight_diagnostics.sort_values(
    "local_deviation",
    ascending=False
).head(25)

Number of midnight observations:
456

Midnight local-deviation summary:
count    456.000000
mean       1.178271
std        2.626930
min        0.000000
50%        0.071224
75%        0.731848
90%        4.023890
95%        8.090596
99%       11.977792
max       13.313825
Name: P_Gaia[kW], dtype: float64

Non-midnight local-deviation summary:
count    131159.000000
mean          0.331585
std           0.472578
min           0.000000
50%           0.136308
75%           0.487230
90%           0.945917
95%           1.285435
99%           2.047547
max           7.995850
Name: P_Gaia[kW], dtype: float64

Midnight observations with local deviation > 1 kW:
92

Midnight observations with local deviation > 2 kW:
68

Midnight observations with local deviation > 3 kW:
55

Midnight observations with local deviation > 5 kW:
40

Midnight observations with local deviation > 8 kW:
24

Midnight observations with local deviation > 10 kW:
14

Largest midnight Gaia discontinuities:


,P_Gaia,previous,next,neighbor_reference,local_deviation
Timestamp,,,,,
2019-01-14 00:00:00+00:00,21.017114,7.694950,7.711627,7.703288,13.313825
2019-02-05 00:00:00+00:00,22.194719,9.204190,9.710230,9.457210,12.737509
2018-09-24 00:00:00+00:00,22.876445,10.508380,10.518813,10.513597,12.362848
2018-12-08 00:00:00+00:00,23.330538,10.959943,11.373940,11.166942,12.163597
2019-03-16 00:00:00+00:00,23.116615,10.263037,11.803360,11.033198,12.083417
2019-01-01 00:00:00+00:00,22.821806,10.403780,11.457090,10.930435,11.891371
2019-03-05 00:00:00+00:00,20.966268,9.857977,8.614417,9.236197,11.730071
2018-09-28 00:00:00+00:00,22.148933,11.206147,10.724177,10.965162,11.183771
2018-12-26 00:00:00+00:00,20.607662,9.398383,9.529870,9.464127,11.143536


### Gaia Midnight-Discontinuity Finding

The midnight Gaia-power anomaly is broader than a simple
`P_Gaia > 12 kW` rule.

Across all 456 midnight observations:

- median local deviation is approximately **0.071 kW**;
- 90th percentile is approximately **4.02 kW**;
- 95th percentile is approximately **8.09 kW**;
- 40 observations differ from their neighbouring values by more than
  5 kW.

In comparison, the non-midnight distribution is much tighter:

- median local deviation is approximately **0.136 kW**;
- 95th percentile is approximately **1.29 kW**;
- 99th percentile is approximately **2.05 kW**.

This indicates that most midnight observations are normal, but a subset
contains unusually large discontinuities that are not explained by the
ordinary short-term variability seen elsewhere in the dataset.

Therefore, neither:

- all midnight observations, nor
- all observations above 12 kW

should automatically be removed.

Because `P_Gaia[kW]` may serve as a forecasting target, no replacement
values are created at this stage. A conservative anomaly flag will first
be defined using the empirical contrast between midnight and
non-midnight local variability.

In [12]:
# ============================================================
# CELL 12 — COMPARE GAIA DISCONTINUITY THRESHOLDS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We need an objective way to distinguish exceptional
# midnight discontinuities from ordinary short-term Gaia
# variability.
#
# Rather than arbitrarily selecting a threshold, we compare
# how frequently different local-deviation thresholds are
# exceeded:
#
#     - at midnight
#     - outside midnight
#
# We also inspect high quantiles of the NON-MIDNIGHT
# distribution, which represents ordinary operation much
# more broadly.
#
# No Gaia values are modified in this cell.


# ------------------------------------------------------------
# 1. HIGH QUANTILES OF ORDINARY NON-MIDNIGHT VARIABILITY
# ------------------------------------------------------------

non_midnight_quantiles = (
    non_midnight_deviation.quantile(
        [
            0.95,
            0.975,
            0.99,
            0.995,
            0.999,
            0.9995,
            0.9999,
        ]
    )
)

print("Non-midnight Gaia local-deviation quantiles:")
print(non_midnight_quantiles)


# ------------------------------------------------------------
# 2. COMPARE FIXED THRESHOLDS
# ------------------------------------------------------------
#
# These remain diagnostic candidates.
#
# For each threshold we calculate:
#
#     count
#     percentage
#
# separately for midnight and non-midnight observations.

thresholds = [
    1,
    2,
    2.5,
    3,
    4,
    5,
    6,
    8,
]


threshold_rows = []

for threshold in thresholds:

    midnight_count = (
        midnight_deviation > threshold
    ).sum()

    non_midnight_count = (
        non_midnight_deviation > threshold
    ).sum()

    midnight_percentage = (
        midnight_count
        / len(midnight_deviation)
        * 100
    )

    non_midnight_percentage = (
        non_midnight_count
        / len(non_midnight_deviation)
        * 100
    )

    threshold_rows.append(
        {
            "threshold_kW": threshold,

            "midnight_count":
                midnight_count,

            "midnight_percent":
                midnight_percentage,

            "non_midnight_count":
                non_midnight_count,

            "non_midnight_percent":
                non_midnight_percentage,
        }
    )


gaia_threshold_comparison = pd.DataFrame(
    threshold_rows
)


# ------------------------------------------------------------
# 3. ADD CONTRAST BETWEEN MIDNIGHT AND ORDINARY OPERATION
# ------------------------------------------------------------
#
# A large ratio means that exceeding the threshold is much
# more characteristic of midnight than ordinary operation.

gaia_threshold_comparison[
    "midnight_to_nonmidnight_rate_ratio"
] = (
    gaia_threshold_comparison[
        "midnight_percent"
    ]
    /
    gaia_threshold_comparison[
        "non_midnight_percent"
    ].replace(0, np.nan)
)


print("\nThreshold comparison:")

gaia_threshold_comparison

Non-midnight Gaia local-deviation quantiles:
0.9500    1.285435
0.9750    1.615727
0.9900    2.047547
0.9950    2.369227
0.9990    3.336602
0.9995    4.084564
0.9999    5.913826
Name: P_Gaia[kW], dtype: float64

Threshold comparison:


,threshold_kW,midnight_count,midnight_percent,non_midnight_count,non_midnight_percent,midnight_to_nonmidnight_rate_ratio
0,1.0,92,20.175439,11751,8.959355,2.251885
1,2.0,68,14.912281,1447,1.103241,13.516792
2,2.5,65,14.254386,510,0.388841,36.658647
3,3.0,55,12.061404,220,0.167735,71.907346
4,4.0,47,10.307018,71,0.054133,190.402551
5,5.0,40,8.771930,38,0.028972,302.767775
6,6.0,34,7.456140,12,0.009149,814.949927
7,8.0,24,5.263158,0,0.000000,NaN


### Gaia Target-Anomaly Criterion

A conservative Gaia-power anomaly criterion was derived from the
empirical distribution of short-term variability rather than from the
turbine's nominal power rating.

The non-midnight local-deviation distribution shows:

- 99.5th percentile: approximately **2.37 kW**
- 99.9th percentile: approximately **3.34 kW**
- 99.95th percentile: approximately **4.08 kW**
- 99.99th percentile: approximately **5.91 kW**

A threshold of **6 kW** therefore lies just above the 99.99th percentile
of ordinary non-midnight behaviour.

Using this threshold:

- 34 of 456 midnight observations exceed 6 kW;
- only 12 of 131,159 non-midnight observations exceed 6 kW;
- the exceedance rate is approximately 815 times greater at midnight.

The conservative anomaly-candidate definition is therefore:

> timestamp is exactly 00:00 UTC **and**
> local `P_Gaia[kW]` deviation from its neighbouring observations
> exceeds 6 kW.

This criterion does not assume that values above the nominal 11–12 kW
range are automatically invalid.

Because `P_Gaia[kW]` may be used as a forecasting target, flagged
observations will not be numerically corrected. The flag can later be
used to exclude anomalous target timestamps from wind-model training
and evaluation.

In [13]:
# ============================================================
# CELL 13 — CREATE CONSERVATIVE GAIA TARGET-ANOMALY FLAG
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 12 established 6 kW as a conservative local-deviation
# threshold because it lies just above the 99.99th percentile
# of ordinary non-midnight Gaia variability.
#
# We define an anomaly candidate only when BOTH conditions hold:
#
#     1. timestamp is exactly 00:00 UTC
#     2. local deviation is greater than 6 kW
#
# IMPORTANT:
#
# No P_Gaia values are changed.
#
# This is only an explicit diagnostic / target-quality flag.


# ------------------------------------------------------------
# 1. DEFINE CONSERVATIVE ANOMALY MASK
# ------------------------------------------------------------

gaia_anomaly_mask = (
    midnight_mask
    &
    (gaia_local_deviation > 6)
)


# ------------------------------------------------------------
# 2. STORE EXPLICIT FLAG
# ------------------------------------------------------------

df_work["gaia_target_anomaly_flag"] = (
    gaia_anomaly_mask.astype("int8")
)


# ------------------------------------------------------------
# 3. VERIFY NUMBER OF FLAGGED OBSERVATIONS
# ------------------------------------------------------------

print("Gaia target-anomaly candidates:")
print(
    df_work["gaia_target_anomaly_flag"].sum()
)


print("\nPercentage of complete dataset:")
print(
    f"{gaia_anomaly_mask.mean() * 100:.6f}%"
)


# ------------------------------------------------------------
# 4. COMPARE WITH THE SIMPLE >12 kW MIDNIGHT RULE
# ------------------------------------------------------------
#
# This helps us verify that the empirically derived rule is
# not simply another way of applying an absolute power limit.

midnight_above_12 = (
    midnight_mask
    &
    (gaia > 12)
)


comparison_counts = pd.Series(
    {
        "Midnight P_Gaia > 12 kW":
            midnight_above_12.sum(),

        "Midnight local deviation > 6 kW":
            gaia_anomaly_mask.sum(),

        "Flagged by both criteria":
            (
                midnight_above_12
                &
                gaia_anomaly_mask
            ).sum(),

        "Only >12 kW criterion":
            (
                midnight_above_12
                &
                ~gaia_anomaly_mask
            ).sum(),

        "Only >6 kW deviation criterion":
            (
                gaia_anomaly_mask
                &
                ~midnight_above_12
            ).sum(),
    }
)

print("\nComparison with absolute >12 kW criterion:")
print(
    comparison_counts
)


# ------------------------------------------------------------
# 5. DISPLAY FLAGGED OBSERVATIONS
# ------------------------------------------------------------

gaia_flagged_examples = pd.DataFrame(
    {
        "P_Gaia":
            gaia.loc[
                gaia_anomaly_mask
            ],

        "previous":
            gaia.shift(1).loc[
                gaia_anomaly_mask
            ],

        "next":
            gaia.shift(-1).loc[
                gaia_anomaly_mask
            ],

        "neighbor_reference":
            gaia_neighbor_reference.loc[
                gaia_anomaly_mask
            ],

        "local_deviation":
            gaia_local_deviation.loc[
                gaia_anomaly_mask
            ],

        "above_12_kW":
            (
                gaia.loc[
                    gaia_anomaly_mask
                ] > 12
            ),
    }
)


print("\nFlagged Gaia target-anomaly candidates:")

gaia_flagged_examples.sort_values(
    "local_deviation",
    ascending=False
)

Gaia target-anomaly candidates:
34

Percentage of complete dataset:
0.025833%

Comparison with absolute >12 kW criterion:
Midnight P_Gaia > 12 kW            35
Midnight local deviation > 6 kW    34
Flagged by both criteria           31
Only >12 kW criterion               4
Only >6 kW deviation criterion      3
dtype: int64

Flagged Gaia target-anomaly candidates:


,P_Gaia,previous,next,neighbor_reference,local_deviation,above_12_kW
Timestamp,,,,,,
2019-01-14 00:00:00+00:00,21.017114,7.694950,7.711627,7.703288,13.313825,True
2019-02-05 00:00:00+00:00,22.194719,9.204190,9.710230,9.457210,12.737509,True
2018-09-24 00:00:00+00:00,22.876445,10.508380,10.518813,10.513597,12.362848,True
2018-12-08 00:00:00+00:00,23.330538,10.959943,11.373940,11.166942,12.163597,True
2019-03-16 00:00:00+00:00,23.116615,10.263037,11.803360,11.033198,12.083417,True
2019-01-01 00:00:00+00:00,22.821806,10.403780,11.457090,10.930435,11.891371,True
2019-03-05 00:00:00+00:00,20.966268,9.857977,8.614417,9.236197,11.730071,True
2018-09-28 00:00:00+00:00,22.148933,11.206147,10.724177,10.965162,11.183771,True
2018-12-26 00:00:00+00:00,20.607662,9.398383,9.529870,9.464127,11.143536,True


### Comparison With an Absolute Power Threshold

The empirical midnight-discontinuity criterion is not equivalent to
simply flagging `P_Gaia > 12 kW`.

The comparison shows:

- 35 midnight observations exceed 12 kW;
- 34 observations satisfy the >6 kW local-deviation criterion;
- 31 observations satisfy both criteria;
- 4 observations exceed 12 kW but do not show sufficiently large local
  discontinuities;
- 3 observations remain below 12 kW but show unusually large midnight
  discontinuities.

This is important because absolute power magnitude alone cannot reliably
identify the suspected daily-boundary artifact.

The local-discontinuity criterion is more consistent with the observed
temporal behaviour of the signal.

The flag is currently treated as an **offline data-quality diagnostic**.
It is not a forecasting input and does not represent an online anomaly
detector, because its diagnostic definition uses both neighbouring
observations.

In [14]:
# ============================================================
# CELL 14 — INSPECT GAIA ANOMALY-CRITERION DISAGREEMENTS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The empirical local-discontinuity criterion and the simple
# >12 kW rule disagree on seven midnight observations:
#
#     - 4 are >12 kW but NOT flagged by local deviation;
#     - 3 are <=12 kW but ARE flagged by local deviation.
#
# Inspecting these cases helps verify that the local temporal
# criterion is behaving sensibly before we finalize it.
#
# No P_Gaia values are modified.


# ------------------------------------------------------------
# 1. DEFINE THE TWO DISAGREEMENT GROUPS
# ------------------------------------------------------------

only_absolute_power_mask = (
    midnight_above_12
    &
    ~gaia_anomaly_mask
)


only_local_deviation_mask = (
    gaia_anomaly_mask
    &
    ~midnight_above_12
)


# ------------------------------------------------------------
# 2. HELPER FUNCTION FOR LOCAL CONTEXT
# ------------------------------------------------------------
#
# For each selected timestamp we show:
#
#     P_Gaia at midnight
#     previous 23:55 observation
#     next 00:05 observation
#     local neighbor reference
#     local deviation
#
# We also include wind speed because P_Gaia physically
# depends strongly on wind speed.

def build_gaia_disagreement_table(mask):

    return pd.DataFrame(
        {
            "P_Gaia":
                df_raw.loc[
                    mask,
                    "P_Gaia[kW]"
                ],

            "previous_P_Gaia":
                df_raw["P_Gaia[kW]"]
                .shift(1)
                .loc[mask],

            "next_P_Gaia":
                df_raw["P_Gaia[kW]"]
                .shift(-1)
                .loc[mask],

            "neighbor_reference":
                gaia_neighbor_reference.loc[
                    mask
                ],

            "local_deviation":
                gaia_local_deviation.loc[
                    mask
                ],

            "wind_speed":
                df_raw.loc[
                    mask,
                    "WIND_SPEED[m1s]"
                ],

            "previous_wind_speed":
                df_raw["WIND_SPEED[m1s]"]
                .shift(1)
                .loc[mask],

            "next_wind_speed":
                df_raw["WIND_SPEED[m1s]"]
                .shift(-1)
                .loc[mask],
        }
    )


# ------------------------------------------------------------
# 3. >12 kW BUT NOT FLAGGED
# ------------------------------------------------------------

only_absolute_power_cases = (
    build_gaia_disagreement_table(
        only_absolute_power_mask
    )
)

print(
    ">12 kW at midnight but NOT flagged "
    "by >6 kW local deviation:"
)

print(
    "\nNumber of cases:",
    len(only_absolute_power_cases)
)

display(
    only_absolute_power_cases
)


# ------------------------------------------------------------
# 4. <=12 kW BUT FLAGGED
# ------------------------------------------------------------

only_local_deviation_cases = (
    build_gaia_disagreement_table(
        only_local_deviation_mask
    )
)

print(
    "\n<=12 kW at midnight but FLAGGED "
    "by >6 kW local deviation:"
)

print(
    "\nNumber of cases:",
    len(only_local_deviation_cases)
)

display(
    only_local_deviation_cases
)

>12 kW at midnight but NOT flagged by >6 kW local deviation:

Number of cases: 4


,P_Gaia,previous_P_Gaia,next_P_Gaia,neighbor_reference,local_deviation,wind_speed,previous_wind_speed,next_wind_speed
Timestamp,,,,,,,,
2018-10-23 00:00:00+00:00,14.673047,8.698253,9.430577,9.064415,5.608632,7.610667,7.146667,7.318000
2018-12-27 00:00:00+00:00,13.343231,7.642450,10.225800,8.934125,4.409106,4.700667,5.238667,6.290000
2019-02-08 00:00:00+00:00,12.529943,7.101667,7.560623,7.331145,5.198798,5.588333,6.035333,6.048333
2019-02-09 00:00:00+00:00,12.508936,6.483727,8.090527,7.287127,5.221810,12.790970,7.292333,6.983333



<=12 kW at midnight but FLAGGED by >6 kW local deviation:

Number of cases: 3


,P_Gaia,previous_P_Gaia,next_P_Gaia,neighbor_reference,local_deviation,wind_speed,previous_wind_speed,next_wind_speed
Timestamp,,,,,,,,
2018-08-09 00:00:00+00:00,10.835505,5.775317,3.166917,4.471117,6.364388,3.546000,3.767667,3.358333
2018-08-10 00:00:00+00:00,9.541628,2.959467,2.264243,2.611855,6.929773,4.947333,4.081333,4.784667
2018-09-30 00:00:00+00:00,11.311214,4.806453,4.296930,4.551692,6.759522,7.654849,4.020333,4.297333


### Need for an Independent Midnight Reference

Inspection of the disagreement cases shows that some midnight records
contain abrupt changes not only in `P_Gaia[kW]` but also in variables
such as wind speed.

Therefore, neighbouring 5-minute observations alone are not sufficient
to determine whether every suspicious midnight Gaia value is corrupted.

The cross-resolution analysis previously established that the provided
5-minute dataset closely follows a `1-minute → 5-minute` processing
lineage.

The 1-minute dataset can therefore provide an independent finer-resolution
reference for evaluating the suspicious 5-minute midnight observations.

The next step reconstructs 5-minute values directly from the 1-minute
dataset and compares them with the provided 5-minute measurements.

No values are modified during this comparison.

In [15]:
# ============================================================
# CELL 15 — VALIDATE MIDNIGHT VALUES USING 1-MINUTE DATA
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# So far, Gaia anomaly investigation has relied on the
# neighbouring 5-minute observations.
#
# However, Cell 14 suggests that multiple variables may
# sometimes be affected at the midnight boundary.
#
# We therefore use the 1-minute dataset as an independent
# finer-resolution reference.
#
# Previous cross-resolution analysis established that the
# provided 5-minute data closely follow:
#
#       1-minute data -> 5-minute mean
#
# This lets us compare the provided midnight observation
# directly with the value reconstructed from its five
# underlying 1-minute observations.
#
# No 5-minute values are modified in this cell.


# ------------------------------------------------------------
# 1. LOAD 1-MINUTE DATASET
# ------------------------------------------------------------

dataset_1min_path = (
    PROJECT_ROOT
    / "solete_dataset"
    / "SOLETE_Pombo_1min.h5"
)

assert dataset_1min_path.exists(), (
    f"1-minute dataset not found: {dataset_1min_path}"
)

df_1min = load_solete(
    dataset_1min_path
)


print("1-minute dataset shape:")
print(df_1min.shape)

print("\n1-minute chronological:")
print(df_1min.index.is_monotonic_increasing)


# ------------------------------------------------------------
# 2. RECONSTRUCT 5-MINUTE VALUES FROM 1-MINUTE DATA
# ------------------------------------------------------------
#
# Each timestamp represents the left edge of the 5-minute bin:
#
#     00:00 -> mean of 00:00, 00:01, ..., 00:04
#
# This is the same reconstruction approach used during the
# cross-resolution analysis.

df_1min_to_5min = (
    df_1min
    .resample("5min")
    .mean()
)


# ------------------------------------------------------------
# 3. ALIGN RECONSTRUCTION WITH PROVIDED 5-MINUTE DATA
# ------------------------------------------------------------

df_1min_to_5min = (
    df_1min_to_5min
    .reindex(df_raw.index)
)

print("\nReconstructed 5-minute shape:")
print(df_1min_to_5min.shape)

print("\nMissing reconstructed timestamps:")
print(
    df_1min_to_5min.isna().all(axis=1).sum()
)


# ------------------------------------------------------------
# 4. CALCULATE MIDNIGHT DIFFERENCES
# ------------------------------------------------------------

gaia_1min_reference = (
    df_1min_to_5min["P_Gaia[kW]"]
)

wind_1min_reference = (
    df_1min_to_5min["WIND_SPEED[m1s]"]
)


gaia_cross_resolution_error = (
    df_raw["P_Gaia[kW]"]
    -
    gaia_1min_reference
).abs()


wind_cross_resolution_error = (
    df_raw["WIND_SPEED[m1s]"]
    -
    wind_1min_reference
).abs()


# ------------------------------------------------------------
# 5. SUMMARIZE MIDNIGHT DIFFERENCES
# ------------------------------------------------------------

print("\nMidnight P_Gaia difference:")
print(
    gaia_cross_resolution_error.loc[
        midnight_mask
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


print("\nMidnight wind-speed difference:")
print(
    wind_cross_resolution_error.loc[
        midnight_mask
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ------------------------------------------------------------
# 6. INSPECT THE SEVEN DISAGREEMENT CASES
# ------------------------------------------------------------
#
# These are:
#
#   - four midnight values >12 kW that were not selected by
#     the >6 kW local-deviation criterion;
#
#   - three values <=12 kW that WERE selected.
#
# The 1-minute reconstruction gives us a much stronger
# independent reference for judging these cases.

disagreement_mask = (
    only_absolute_power_mask
    |
    only_local_deviation_mask
)


cross_resolution_disagreement = pd.DataFrame(
    {
        "provided_5min_P_Gaia":
            df_raw.loc[
                disagreement_mask,
                "P_Gaia[kW]"
            ],

        "reconstructed_1min_P_Gaia":
            gaia_1min_reference.loc[
                disagreement_mask
            ],

        "P_Gaia_abs_difference":
            gaia_cross_resolution_error.loc[
                disagreement_mask
            ],

        "provided_5min_wind_speed":
            df_raw.loc[
                disagreement_mask,
                "WIND_SPEED[m1s]"
            ],

        "reconstructed_1min_wind_speed":
            wind_1min_reference.loc[
                disagreement_mask
            ],

        "wind_speed_abs_difference":
            wind_cross_resolution_error.loc[
                disagreement_mask
            ],

        "above_12_kW":
            (
                df_raw.loc[
                    disagreement_mask,
                    "P_Gaia[kW]"
                ]
                > 12
            ),

        "local_deviation_flag":
            gaia_anomaly_mask[
                disagreement_mask
            ],
    }
)


print(
    "\nCross-resolution comparison for "
    "the seven disagreement cases:"
)

cross_resolution_disagreement

1-minute dataset shape:
(658081, 11)

1-minute chronological:
True

Reconstructed 5-minute shape:
(131617, 11)

Missing reconstructed timestamps:
0

Midnight P_Gaia difference:
count    458.000000
mean       0.813100
std        2.104043
min        0.000000
50%        0.000000
75%        0.000099
90%        3.496335
95%        6.380515
99%        9.361824
max        9.909903
Name: P_Gaia[kW], dtype: float64

Midnight wind-speed difference:
count    4.580000e+02
mean     9.409765e-01
std      1.722350e+00
min      0.000000e+00
50%      4.440892e-16
75%      1.199728e+00
90%      3.280748e+00
95%      4.331682e+00
99%      7.588840e+00
max      9.680528e+00
Name: WIND_SPEED[m1s], dtype: float64

Cross-resolution comparison for the seven disagreement cases:


,provided_5min_P_Gaia,reconstructed_1min_P_Gaia,P_Gaia_abs_difference,provided_5min_wind_speed,reconstructed_1min_wind_speed,wind_speed_abs_difference,above_12_kW,local_deviation_flag
Timestamp,,,,,,,,
2018-08-09 00:00:00+00:00,10.835505,6.306107,4.529398,3.546000,3.546000,4.440892e-16,False,True
2018-08-10 00:00:00+00:00,9.541628,2.916593,6.625035,4.947333,4.947333,0.000000e+00,False,True
2018-09-30 00:00:00+00:00,11.311214,6.323122,4.988092,7.654849,4.535644,3.119205e+00,False,True
2018-10-23 00:00:00+00:00,14.673047,9.835807,4.837240,7.610667,7.610667,8.881784e-16,True,False
2018-12-27 00:00:00+00:00,13.343231,8.285265,5.057966,4.700667,4.700667,0.000000e+00,True,False
2019-02-08 00:00:00+00:00,12.529943,8.317955,4.211988,5.588333,5.588333,0.000000e+00,True,False
2019-02-09 00:00:00+00:00,12.508936,7.267464,5.241473,12.790970,8.628124,4.162846e+00,True,False


### Cross-Resolution Validation of Gaia Midnight Anomalies

The 1-minute dataset provides an independent finer-resolution reference
for evaluating suspicious 5-minute midnight observations.

The seven cases on which the absolute-power and local-discontinuity
criteria disagreed all show substantial discrepancies between the
provided 5-minute `P_Gaia[kW]` value and the value reconstructed from
the corresponding 1-minute observations.

The absolute differences range from approximately **4.21 to 6.63 kW**.

This includes:

- all three values below 12 kW that were identified by the local
  discontinuity criterion;
- all four values above 12 kW that were not identified by the
  >6 kW local-discontinuity criterion.

Therefore, the previously defined 34-observation local-discontinuity
flag should be treated as a useful diagnostic rather than the final
Gaia target-quality rule.

The cross-resolution comparison is stronger evidence because it compares
the provided 5-minute observation with an independently reconstructed
value from the finer-resolution dataset belonging to the same practical
processing lineage.

Some cases also show that `P_Gaia[kW]` can disagree substantially even
when the corresponding provided and reconstructed wind speeds agree
almost exactly. This suggests that the midnight issue is not always
explained solely by a wind-speed anomaly.

No Gaia values are modified yet. The full cross-resolution error
distribution will next be examined before defining the final target
quality flag.

In [16]:
# ============================================================
# CELL 16 — CHARACTERIZE GAIA CROSS-RESOLUTION ERRORS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 15 showed that the 1-minute -> 5-minute reconstruction
# provides stronger evidence for suspicious midnight Gaia
# observations than the neighbour-only heuristic.
#
# We now compare the FULL distribution of reconstruction error:
#
#     | provided 5-minute P_Gaia
#       - reconstructed 1-minute P_Gaia |
#
# separately for:
#
#     - midnight observations
#     - non-midnight observations
#
# This will let us determine whether there is a natural
# empirical separation between ordinary cross-resolution
# differences and the suspected midnight artifact.
#
# No values are modified.


# ------------------------------------------------------------
# 1. SPLIT CROSS-RESOLUTION ERRORS
# ------------------------------------------------------------

gaia_midnight_xres_error = (
    gaia_cross_resolution_error
    .loc[midnight_mask]
    .dropna()
)

gaia_nonmidnight_xres_error = (
    gaia_cross_resolution_error
    .loc[~midnight_mask]
    .dropna()
)


# ------------------------------------------------------------
# 2. SUMMARY DISTRIBUTIONS
# ------------------------------------------------------------

percentiles = [
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    0.995,
    0.999,
    0.9995,
    0.9999,
]


print("MIDNIGHT P_Gaia cross-resolution error:")
print(
    gaia_midnight_xres_error.describe(
        percentiles=percentiles
    )
)


print("\nNON-MIDNIGHT P_Gaia cross-resolution error:")
print(
    gaia_nonmidnight_xres_error.describe(
        percentiles=percentiles
    )
)


# ------------------------------------------------------------
# 3. COMPARE ERROR THRESHOLDS
# ------------------------------------------------------------
#
# Small thresholds are included because most provided 5-minute
# observations match the 1-minute reconstruction extremely
# closely.
#
# These thresholds are diagnostic only at this stage.

xres_thresholds = [
    0.001,
    0.01,
    0.1,
    0.5,
    1.0,
    2.0,
    3.0,
    4.0,
    5.0,
]


xres_threshold_rows = []

for threshold in xres_thresholds:

    midnight_count = (
        gaia_midnight_xres_error > threshold
    ).sum()

    nonmidnight_count = (
        gaia_nonmidnight_xres_error > threshold
    ).sum()

    midnight_percent = (
        midnight_count
        / len(gaia_midnight_xres_error)
        * 100
    )

    nonmidnight_percent = (
        nonmidnight_count
        / len(gaia_nonmidnight_xres_error)
        * 100
    )

    xres_threshold_rows.append(
        {
            "threshold_kW": threshold,

            "midnight_count":
                midnight_count,

            "midnight_percent":
                midnight_percent,

            "nonmidnight_count":
                nonmidnight_count,

            "nonmidnight_percent":
                nonmidnight_percent,
        }
    )


gaia_xres_threshold_comparison = pd.DataFrame(
    xres_threshold_rows
)


# ------------------------------------------------------------
# 4. CALCULATE MIDNIGHT / NON-MIDNIGHT RATE CONTRAST
# ------------------------------------------------------------

gaia_xres_threshold_comparison[
    "midnight_to_nonmidnight_rate_ratio"
] = (
    gaia_xres_threshold_comparison[
        "midnight_percent"
    ]
    /
    gaia_xres_threshold_comparison[
        "nonmidnight_percent"
    ].replace(0, np.nan)
)


print("\nCross-resolution threshold comparison:")

gaia_xres_threshold_comparison

MIDNIGHT P_Gaia cross-resolution error:
count     458.000000
mean        0.813100
std         2.104043
min         0.000000
50%         0.000000
75%         0.000099
90%         3.496335
95%         6.380515
99%         9.361824
99.5%       9.581109
99.9%       9.810827
99.95%      9.860365
99.99%      9.899996
max         9.909903
Name: P_Gaia[kW], dtype: float64

NON-MIDNIGHT P_Gaia cross-resolution error:
count     1.311590e+05
mean      1.143529e-04
std       1.275817e-02
min       0.000000e+00
50%       0.000000e+00
75%       1.387779e-17
90%       8.881784e-16
95%       8.881784e-16
99%       1.776357e-15
99.5%     1.776357e-15
99.9%     1.776357e-15
99.95%    1.776357e-15
99.99%    3.244954e-01
max       2.656057e+00
Name: P_Gaia[kW], dtype: float64

Cross-resolution threshold comparison:


,threshold_kW,midnight_count,midnight_percent,nonmidnight_count,nonmidnight_percent,midnight_to_nonmidnight_rate_ratio
0,0.001,98,21.397380,31,0.023635,905.309339
1,0.010,92,20.087336,29,0.022111,908.494805
2,0.100,92,20.087336,19,0.014486,1386.649966
3,0.500,85,18.558952,7,0.005337,3477.390830
4,1.000,74,16.157205,5,0.003812,4238.325764
5,2.000,60,13.100437,3,0.002287,5727.467249
6,3.000,48,10.480349,0,0.000000,NaN
7,4.000,42,9.170306,0,0.000000,NaN
8,5.000,34,7.423581,0,0.000000,NaN


### Final Gaia Target-Quality Criterion

Cross-resolution validation provides substantially stronger evidence for
the midnight Gaia-power artifact than either an absolute power threshold
or a neighbouring-observation heuristic.

For the absolute difference between:

- provided 5-minute `P_Gaia[kW]`, and
- `P_Gaia[kW]` reconstructed from the 1-minute dataset,

the non-midnight observations show extremely close agreement.

Notably:

- the maximum non-midnight difference is approximately **2.66 kW**;
- no non-midnight observation exceeds **3 kW**;
- 48 midnight observations exceed **3 kW**;
- 42 midnight observations exceed **4 kW**;
- 34 midnight observations exceed **5 kW**.

A cross-resolution error threshold of **3 kW** is therefore selected as
the primary dataset-specific Gaia target-quality criterion.

The final criterion is:

> timestamp is exactly 00:00 UTC **and**
> the absolute difference between provided 5-minute `P_Gaia[kW]` and
> the value reconstructed from the 1-minute dataset exceeds 3 kW.

This threshold is not interpreted as a physical turbine limit. It is an
empirical data-quality criterion derived from the observed processing
relationship between the 1-minute and 5-minute SOLETE datasets.

The affected Gaia values will **not be numerically corrected** because
`P_Gaia[kW]` may serve as a forecasting target. Instead, the timestamps
will be explicitly flagged so that they can later be excluded from
wind-target training and evaluation.

The cross-resolution reference is used only for offline data-quality
assessment and will never be supplied to the forecasting model as an
input feature.

In [17]:
# ============================================================
# CELL 17 — FINALIZE GAIA TARGET-QUALITY FLAGS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We now distinguish between two different Gaia diagnostics:
#
#   1. Local-discontinuity flag
#      --------------------------------
#      Midnight observation whose difference from the mean of
#      its neighbouring 5-minute values exceeds 6 kW.
#
#      This was useful during exploratory diagnosis.
#
#
#   2. Final cross-resolution quality flag
#      -------------------------------------
#      Midnight observation whose provided 5-minute P_Gaia
#      differs from the independently reconstructed 1-minute
#      -> 5-minute value by more than 3 kW.
#
#      This is our preferred dataset-quality criterion.
#
#
# IMPORTANT:
#
# P_Gaia[kW] itself is NOT modified.


# ============================================================
# 1. RENAME / PRESERVE THE EARLIER LOCAL DIAGNOSTIC
# ============================================================
#
# Cell 13 originally stored the neighbour-based flag under
# gaia_target_anomaly_flag.
#
# We preserve its information but give it a more precise name.

df_work["gaia_local_discontinuity_flag"] = (
    df_work["gaia_target_anomaly_flag"].copy()
)


# ============================================================
# 2. CREATE FINAL CROSS-RESOLUTION QUALITY MASK
# ============================================================

gaia_xres_quality_mask = (
    midnight_mask
    &
    (gaia_cross_resolution_error > 3.0)
)


# ============================================================
# 3. STORE FINAL QUALITY FLAG
# ============================================================

df_work["gaia_target_quality_flag"] = (
    gaia_xres_quality_mask.astype("int8")
)


# ============================================================
# 4. VERIFY COUNTS
# ============================================================

print("Local-discontinuity diagnostic flags:")
print(
    df_work["gaia_local_discontinuity_flag"].sum()
)

print("\nFinal cross-resolution Gaia quality flags:")
print(
    df_work["gaia_target_quality_flag"].sum()
)

print("\nPercentage of complete 5-minute dataset flagged:")
print(
    f"{gaia_xres_quality_mask.mean() * 100:.6f}%"
)


# ============================================================
# 5. COMPARE THE TWO DIAGNOSTIC RULES
# ============================================================

local_mask = (
    df_work["gaia_local_discontinuity_flag"] == 1
)

comparison = pd.Series(
    {
        "Local >6 kW discontinuity flag":
            local_mask.sum(),

        "Cross-resolution >3 kW quality flag":
            gaia_xres_quality_mask.sum(),

        "Flagged by both":
            (
                local_mask
                &
                gaia_xres_quality_mask
            ).sum(),

        "Only local-discontinuity flag":
            (
                local_mask
                &
                ~gaia_xres_quality_mask
            ).sum(),

        "Only cross-resolution quality flag":
            (
                gaia_xres_quality_mask
                &
                ~local_mask
            ).sum(),
    }
)

print("\nComparison of Gaia quality criteria:")
print(comparison)


# ============================================================
# 6. CONFIRM TARGET VALUES ARE UNCHANGED
# ============================================================

print("\nRaw and working P_Gaia identical?")
print(
    df_raw["P_Gaia[kW]"].equals(
        df_work["P_Gaia[kW]"]
    )
)


# ============================================================
# 7. SHOW FINAL FLAGGED EXAMPLES
# ============================================================

gaia_final_quality_examples = pd.DataFrame(
    {
        "provided_5min_P_Gaia":
            df_raw.loc[
                gaia_xres_quality_mask,
                "P_Gaia[kW]"
            ],

        "reconstructed_1min_P_Gaia":
            gaia_1min_reference.loc[
                gaia_xres_quality_mask
            ],

        "cross_resolution_error":
            gaia_cross_resolution_error.loc[
                gaia_xres_quality_mask
            ],

        "local_deviation":
            gaia_local_deviation.loc[
                gaia_xres_quality_mask
            ],

        "local_discontinuity_flag":
            df_work.loc[
                gaia_xres_quality_mask,
                "gaia_local_discontinuity_flag"
            ],
    }
)


print("\nLargest final Gaia target-quality cases:")

gaia_final_quality_examples.sort_values(
    "cross_resolution_error",
    ascending=False
).head(20)

Local-discontinuity diagnostic flags:
34

Final cross-resolution Gaia quality flags:
48

Percentage of complete 5-minute dataset flagged:
0.036469%

Comparison of Gaia quality criteria:
Local >6 kW discontinuity flag         34
Cross-resolution >3 kW quality flag    48
Flagged by both                        34
Only local-discontinuity flag           0
Only cross-resolution quality flag     14
dtype: int64

Raw and working P_Gaia identical?
True

Largest final Gaia target-quality cases:


,provided_5min_P_Gaia,reconstructed_1min_P_Gaia,cross_resolution_error,local_deviation,local_discontinuity_flag
Timestamp,,,,,
2019-01-14 00:00:00+00:00,21.017114,11.107210,9.909903,13.313825,1
2018-12-08 00:00:00+00:00,23.330538,13.637433,9.693105,12.163597,1
2019-01-01 00:00:00+00:00,22.821806,13.221544,9.600262,11.891371,1
2019-02-05 00:00:00+00:00,22.194719,12.661661,9.533058,12.737509,1
2018-09-24 00:00:00+00:00,22.876445,13.386991,9.489454,12.362848,1
2018-12-26 00:00:00+00:00,20.607662,11.342120,9.265542,11.143536,1
2019-03-16 00:00:00+00:00,23.116615,13.974340,9.142275,12.083417,1
2019-03-05 00:00:00+00:00,20.966268,11.965749,9.000519,11.730071,1
2018-08-12 00:00:00+00:00,21.930482,12.937161,8.993321,11.133907,1


### Final Gaia Target-Quality Outcome

Cross-resolution validation identifies **48** 5-minute Gaia-power
observations as target-quality anomalies.

The earlier local-discontinuity diagnostic identified 34 observations,
and all 34 are contained within the final 48-observation
cross-resolution set.

An additional 14 observations are detected only by the stronger
1-minute-to-5-minute comparison.

Therefore:

- the neighbour-based rule is retained only as a diagnostic;
- `gaia_target_quality_flag` is the preferred offline data-quality flag;
- `P_Gaia[kW]` itself remains completely unchanged;
- flagged timestamps can later be excluded when training or evaluating
  models whose target is `P_Gaia[kW]`;
- the flag will not be supplied to the forecasting model as an input
  feature.

The final flag affects only approximately **0.0365%** of the complete
5-minute dataset.

Because its definition uses the finer-resolution dataset,
`gaia_target_quality_flag` is specifically an offline dataset-curation
tool rather than a general preprocessing rule for unseen future data.

In [18]:
# ============================================================
# CELL 18 — FINALIZE PREPROCESSING WORKING TABLE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# During preprocessing investigation we created several
# temporary candidate columns.
#
# Now that the decisions are finalized, we remove only the
# obsolete candidates and retain:
#
# ORIGINAL SOURCE COLUMNS
#     - all 11 original SOLETE variables
#
# FINAL / USEFUL PREPROCESSING COLUMNS
#     - pressure_anomaly_flag
#     - Pressure_clean[mbar]
#     - humidity_anomaly_flag
#     - Humidity_clean
#     - wind_dir_wrap_flag
#     - WIND_DIR_wrapped[deg]
#     - wind_dir_sin
#     - wind_dir_cos
#     - gaia_local_discontinuity_flag
#     - gaia_target_quality_flag
#
# The Gaia local-discontinuity flag is retained for diagnostic
# traceability, while gaia_target_quality_flag is the final
# offline target-quality mask.


# ------------------------------------------------------------
# 1. IDENTIFY TEMPORARY COLUMNS THAT ARE NO LONGER NEEDED
# ------------------------------------------------------------

temporary_columns = [
    "Pressure_minus1000[mbar]",
    "Pressure_ffill[mbar]",
    "gaia_target_anomaly_flag",
]


# ------------------------------------------------------------
# 2. REMOVE ONLY COLUMNS THAT CURRENTLY EXIST
# ------------------------------------------------------------

columns_to_drop = [
    column
    for column in temporary_columns
    if column in df_work.columns
]

df_work = df_work.drop(
    columns=columns_to_drop
)


# ------------------------------------------------------------
# 3. VERIFY FINAL TABLE STRUCTURE
# ------------------------------------------------------------

print("Dataset shape after preprocessing:")
print(df_work.shape)

print("\nColumns:")
for i, column in enumerate(
    df_work.columns,
    start=1
):
    print(f"{i:2d}. {column}")


# ------------------------------------------------------------
# 4. VERIFY FINAL CLEANED VARIABLES
# ------------------------------------------------------------

print("\nFinal cleaned pressure range:")
print(
    df_work["Pressure_clean[mbar]"].min(),
    "to",
    df_work["Pressure_clean[mbar]"].max()
)

print("\nFinal cleaned humidity range:")
print(
    df_work["Humidity_clean"].min(),
    "to",
    df_work["Humidity_clean"].max()
)


# ------------------------------------------------------------
# 5. VERIFY QUALITY / DIAGNOSTIC FLAG COUNTS
# ------------------------------------------------------------

flag_columns = [
    "pressure_anomaly_flag",
    "humidity_anomaly_flag",
    "wind_dir_wrap_flag",
    "gaia_local_discontinuity_flag",
    "gaia_target_quality_flag",
]

print("\nFlag counts:")

for column in flag_columns:
    print(
        f"{column}: "
        f"{int(df_work[column].sum())}"
    )


# ------------------------------------------------------------
# 6. VERIFY THERE ARE NO MISSING VALUES
# ------------------------------------------------------------

print("\nTotal missing values in df_work:")
print(
    df_work.isna().sum().sum()
)


# ------------------------------------------------------------
# 7. VERIFY ORIGINAL SOURCE COLUMNS STILL MATCH df_raw
# ------------------------------------------------------------

original_columns_unchanged = all(
    df_raw[column].equals(
        df_work[column]
    )
    for column in df_raw.columns
)

print("\nAll original source columns unchanged?")
print(original_columns_unchanged)

Dataset shape after preprocessing:
(131617, 21)

Columns:
 1. TEMPERATURE[degC]
 2. HUMIDITY[%]
 3. WIND_SPEED[m1s]
 4. WIND_DIR[deg]
 5. GHI[kW1m2]
 6. POA Irr[kW1m2]
 7. P_Gaia[kW]
 8. P_Solar[kW]
 9. Pressure[mbar]
10. Azimuth[deg]
11. Elevation[deg]
12. pressure_anomaly_flag
13. Pressure_clean[mbar]
14. wind_dir_wrap_flag
15. WIND_DIR_wrapped[deg]
16. wind_dir_sin
17. wind_dir_cos
18. humidity_anomaly_flag
19. Humidity_clean
20. gaia_local_discontinuity_flag
21. gaia_target_quality_flag

Final cleaned pressure range:
974.2629986572266 to 1042.3197102864617

Final cleaned humidity range:
0.0 to 1.0

Flag counts:
pressure_anomaly_flag: 456
humidity_anomaly_flag: 164
wind_dir_wrap_flag: 103
gaia_local_discontinuity_flag: 34
gaia_target_quality_flag: 48

Total missing values in df_work:
0

All original source columns unchanged?
True


## Forecast-Time Feature Availability and Leakage Control

Before creating forecasting features, each variable must be classified
according to whether it would realistically be available when a forecast
is issued.

For the primary baseline experiments, the following assumption is used:

> At forecast origin `t`, measurements observed at or before `t` may be
> used as historical information. Measurements after `t` are unavailable
> unless they are deterministic quantities that can be calculated in
> advance.

Therefore:

- historical meteorological measurements may be used through lagged or
  window-based features;
- historical target values may be used through lagged features;
- future measured GHI, POA irradiance, wind speed, temperature, humidity,
  pressure, or power must not be supplied directly to the model;
- calendar variables derived from future timestamps are known in advance;
- astronomical variables calculated deterministically from timestamp and
  location can also be known in advance;
- preprocessing anomaly flags are quality-control information and are not
  default forecasting inputs.

This distinction prevents information leakage and allows later model
comparisons to remain realistic.

If external weather or irradiance forecasts are introduced later, they
will be evaluated as a separate forecasting scenario rather than being
silently treated as known future measurements.

In [19]:
# ============================================================
# CELL 19 — FORECAST-TIME FEATURE AVAILABILITY REGISTER
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# This register documents how each current variable may be
# used in forecasting.
#
# We assume a forecast is issued at time t for one or more
# future timestamps t+h.
#
# Under the baseline forecasting setup:
#
#   - observations at or before t may be used historically;
#   - observations after t are NOT known;
#   - deterministic calendar / astronomical quantities can
#     be computed for future timestamps;
#   - quality-control flags are not model inputs by default.
#
# This cell does NOT modify df_work.


feature_availability_register = pd.DataFrame(
    [
        # ----------------------------------------------------
        # METEOROLOGICAL / PHYSICAL MEASUREMENTS
        # ----------------------------------------------------

        {
            "variable": "TEMPERATURE[degC]",
            "role": "meteorological measurement",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Future measured temperature is unknown unless an "
                "external weather forecast is explicitly supplied."
            ),
        },

        {
            "variable": "Humidity_clean",
            "role": "meteorological measurement",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Use cleaned historical humidity; future measured "
                "humidity is unavailable in the baseline setup."
            ),
        },

        {
            "variable": "WIND_SPEED[m1s]",
            "role": "meteorological measurement",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Future measured wind speed would create leakage "
                "unless a separate wind forecast is available."
            ),
        },

        {
            "variable": "wind_dir_sin",
            "role": "meteorological circular feature",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Circular wind-direction representation; future "
                "measured direction is not known in advance."
            ),
        },

        {
            "variable": "wind_dir_cos",
            "role": "meteorological circular feature",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Circular wind-direction representation; future "
                "measured direction is not known in advance."
            ),
        },

        {
            "variable": "Pressure_clean[mbar]",
            "role": "meteorological measurement",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Use cleaned historical pressure; future measured "
                "pressure is unavailable in the baseline setup."
            ),
        },

        {
            "variable": "GHI[kW1m2]",
            "role": "irradiance measurement",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Same-timestamp GHI is highly predictive of PV power "
                "but future measured GHI would cause leakage."
            ),
        },

        {
            "variable": "POA Irr[kW1m2]",
            "role": "irradiance measurement",
            "historical_use": "Yes",
            "future_direct_use": "No",
            "default_model_use": "Lagged/history only",
            "reason": (
                "Future measured POA irradiance is not available at "
                "forecast origin and must not be used directly."
            ),
        },


        # ----------------------------------------------------
        # FORECAST TARGETS
        # ----------------------------------------------------

        {
            "variable": "P_Solar[kW]",
            "role": "solar forecasting target",
            "historical_use": "Yes",
            "future_direct_use": "Target only",
            "default_model_use": "Lagged target / prediction target",
            "reason": (
                "Past solar power may be used as autoregressive "
                "information; future solar power is what is predicted."
            ),
        },

        {
            "variable": "P_Gaia[kW]",
            "role": "wind forecasting target",
            "historical_use": "Yes",
            "future_direct_use": "Target only",
            "default_model_use": "Lagged target / prediction target",
            "reason": (
                "Past wind power may be used autoregressively; future "
                "wind power is the prediction target."
            ),
        },


        # ----------------------------------------------------
        # DATASET-PROVIDED SOLAR POSITION VARIABLES
        # ----------------------------------------------------

        {
            "variable": "Azimuth[deg]",
            "role": "dataset astronomical variable",
            "historical_use": "Diagnostic",
            "future_direct_use": "Not by default",
            "default_model_use": "Do not use yet",
            "reason": (
                "Dataset representation showed unusual behaviour. "
                "Astronomical features will instead be computed "
                "deterministically from timestamp and site location."
            ),
        },

        {
            "variable": "Elevation[deg]",
            "role": "dataset astronomical variable",
            "historical_use": "Diagnostic",
            "future_direct_use": "Not by default",
            "default_model_use": "Do not use yet",
            "reason": (
                "Dataset elevation uses a nonstandard/clipped "
                "representation; astronomical features will later "
                "be computed independently."
            ),
        },


        # ----------------------------------------------------
        # QUALITY-CONTROL / DIAGNOSTIC VARIABLES
        # ----------------------------------------------------

        {
            "variable": "pressure_anomaly_flag",
            "role": "quality-control flag",
            "historical_use": "Diagnostic",
            "future_direct_use": "No",
            "default_model_use": "Exclude",
            "reason": (
                "Documents preprocessing provenance rather than "
                "physical forecasting information."
            ),
        },

        {
            "variable": "humidity_anomaly_flag",
            "role": "quality-control flag",
            "historical_use": "Diagnostic",
            "future_direct_use": "No",
            "default_model_use": "Exclude",
            "reason": (
                "Documents corrected humidity observations and is "
                "not intended as a forecasting feature."
            ),
        },

        {
            "variable": "wind_dir_wrap_flag",
            "role": "representation diagnostic",
            "historical_use": "Diagnostic",
            "future_direct_use": "No",
            "default_model_use": "Exclude",
            "reason": (
                "Indicates source angular representation rather than "
                "physical wind behaviour."
            ),
        },

        {
            "variable": "gaia_local_discontinuity_flag",
            "role": "offline diagnostic flag",
            "historical_use": "Diagnostic",
            "future_direct_use": "No",
            "default_model_use": "Exclude",
            "reason": (
                "Uses neighbouring observations and is retained only "
                "for data-quality traceability."
            ),
        },

        {
            "variable": "gaia_target_quality_flag",
            "role": "offline target-quality mask",
            "historical_use": "Quality control",
            "future_direct_use": "No",
            "default_model_use": "Exclude from feature vector",
            "reason": (
                "Used later to exclude questionable Gaia target "
                "timestamps from wind-model training/evaluation."
            ),
        },
    ]
)


# ------------------------------------------------------------
# DISPLAY THE REGISTER
# ------------------------------------------------------------

pd.set_option(
    "display.max_colwidth",
    None
)

feature_availability_register

,variable,role,historical_use,future_direct_use,default_model_use,reason
0,TEMPERATURE[degC],meteorological measurement,Yes,No,Lagged/history only,Future measured temperature is unknown unless an external weather forecast is explicitly supplied.
1,Humidity_clean,meteorological measurement,Yes,No,Lagged/history only,Use cleaned historical humidity; future measured humidity is unavailable in the baseline setup.
2,WIND_SPEED[m1s],meteorological measurement,Yes,No,Lagged/history only,Future measured wind speed would create leakage unless a separate wind forecast is available.
3,wind_dir_sin,meteorological circular feature,Yes,No,Lagged/history only,Circular wind-direction representation; future measured direction is not known in advance.
4,wind_dir_cos,meteorological circular feature,Yes,No,Lagged/history only,Circular wind-direction representation; future measured direction is not known in advance.
5,Pressure_clean[mbar],meteorological measurement,Yes,No,Lagged/history only,Use cleaned historical pressure; future measured pressure is unavailable in the baseline setup.
6,GHI[kW1m2],irradiance measurement,Yes,No,Lagged/history only,Same-timestamp GHI is highly predictive of PV power but future measured GHI would cause leakage.
7,POA Irr[kW1m2],irradiance measurement,Yes,No,Lagged/history only,Future measured POA irradiance is not available at forecast origin and must not be used directly.
8,P_Solar[kW],solar forecasting target,Yes,Target only,Lagged target / prediction target,Past solar power may be used as autoregressive information; future solar power is what is predicted.
9,P_Gaia[kW],wind forecasting target,Yes,Target only,Lagged target / prediction target,Past wind power may be used autoregressively; future wind power is the prediction target.


## Gregorian Calendar and Time Features

The first deterministic feature group is derived directly from the
timestamp.

The SOLETE timestamps are stored in UTC. UTC is used as the primary
calendar-time reference because it:

- preserves the exact continuous 5-minute timeline;
- avoids daylight-saving-time gaps and repeated local timestamps;
- is known exactly for every future forecast timestamp;
- provides a reproducible baseline before astronomical and Vedic
  calendar information is introduced.

The raw integer calendar components created below are primarily useful
for interpretation and verification.

Periodic quantities such as time of day and day of year will later be
converted to sine/cosine representations before being used as default
model features.

In [20]:
# ============================================================
# CELL 20 — EXTRACT BASIC UTC GREGORIAN FEATURES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Calendar features are deterministic:
#
# if we know a future forecast timestamp, we also know its:
#
#     year
#     month
#     day
#     day of week
#     day of year
#     hour
#     minute
#
# Therefore these features can be calculated for future
# timestamps without information leakage.
#
# This cell extracts the basic UTC calendar components.
#
# We do NOT create cyclical sine/cosine encodings yet.
# That will be handled separately after verification.


# ------------------------------------------------------------
# 1. CALENDAR DATE COMPONENTS
# ------------------------------------------------------------

df_work["utc_year"] = (
    df_work.index.year
)

df_work["utc_month"] = (
    df_work.index.month
)

df_work["utc_day"] = (
    df_work.index.day
)


# ------------------------------------------------------------
# 2. WEEK-BASED COMPONENT
# ------------------------------------------------------------
#
# pandas convention:
#
#     Monday    = 0
#     Tuesday   = 1
#     ...
#     Sunday    = 6

df_work["utc_day_of_week"] = (
    df_work.index.dayofweek
)


# ------------------------------------------------------------
# 3. ANNUAL POSITION
# ------------------------------------------------------------
#
# dayofyear ranges from:
#
#     1 ... 365
#
# or 366 in a leap year.

df_work["utc_day_of_year"] = (
    df_work.index.dayofyear
)


# ------------------------------------------------------------
# 4. INTRADAY COMPONENTS
# ------------------------------------------------------------

df_work["utc_hour"] = (
    df_work.index.hour
)

df_work["utc_minute"] = (
    df_work.index.minute
)


# ------------------------------------------------------------
# 5. MINUTE OF DAY
# ------------------------------------------------------------
#
# This expresses intraday position on a single continuous
# scale:
#
#     00:00 ->    0
#     01:00 ->   60
#     12:00 ->  720
#     23:55 -> 1435
#
# Because the dataset has 5-minute resolution, expected values
# occur in steps of five.

df_work["utc_minute_of_day"] = (
    df_work["utc_hour"] * 60
    +
    df_work["utc_minute"]
)


# ------------------------------------------------------------
# 6. WEEKEND INDICATOR
# ------------------------------------------------------------
#
# Saturday = 5
# Sunday   = 6

df_work["utc_is_weekend"] = (
    df_work["utc_day_of_week"]
    .isin([5, 6])
    .astype("int8")
)


# ------------------------------------------------------------
# 7. VERIFY RANGES
# ------------------------------------------------------------

calendar_columns = [
    "utc_year",
    "utc_month",
    "utc_day",
    "utc_day_of_week",
    "utc_day_of_year",
    "utc_hour",
    "utc_minute",
    "utc_minute_of_day",
    "utc_is_weekend",
]


print("Gregorian UTC feature ranges:\n")

for column in calendar_columns:

    print(
        f"{column:20s}: "
        f"{df_work[column].min()} "
        f"to "
        f"{df_work[column].max()}"
    )


# ------------------------------------------------------------
# 8. VERIFY 5-MINUTE CLOCK POSITIONS
# ------------------------------------------------------------

print("\nUnique UTC minute values:")
print(
    sorted(
        df_work["utc_minute"].unique()
    )
)


# ------------------------------------------------------------
# 9. DISPLAY SAMPLE FEATURES
# ------------------------------------------------------------

print("\nFirst 12 timestamp-derived rows:")

df_work[
    calendar_columns
].head(12)

Gregorian UTC feature ranges:

utc_year            : 2018 to 2019
utc_month           : 1 to 12
utc_day             : 1 to 31
utc_day_of_week     : 0 to 6
utc_day_of_year     : 1 to 365
utc_hour            : 0 to 23
utc_minute          : 0 to 55
utc_minute_of_day   : 0 to 1435
utc_is_weekend      : 0 to 1

Unique UTC minute values:
[np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25), np.int32(30), np.int32(35), np.int32(40), np.int32(45), np.int32(50), np.int32(55)]

First 12 timestamp-derived rows:


,utc_year,utc_month,utc_day,utc_day_of_week,utc_day_of_year,utc_hour,utc_minute,utc_minute_of_day,utc_is_weekend
Timestamp,,,,,,,,,
2018-06-01 00:00:00+00:00,2018,6,1,4,152,0,0,0,0
2018-06-01 00:05:00+00:00,2018,6,1,4,152,0,5,5,0
2018-06-01 00:10:00+00:00,2018,6,1,4,152,0,10,10,0
2018-06-01 00:15:00+00:00,2018,6,1,4,152,0,15,15,0
2018-06-01 00:20:00+00:00,2018,6,1,4,152,0,20,20,0
2018-06-01 00:25:00+00:00,2018,6,1,4,152,0,25,25,0
2018-06-01 00:30:00+00:00,2018,6,1,4,152,0,30,30,0
2018-06-01 00:35:00+00:00,2018,6,1,4,152,0,35,35,0
2018-06-01 00:40:00+00:00,2018,6,1,4,152,0,40,40,0


### Cyclical Encoding of Gregorian Time

Raw calendar integers do not naturally represent periodic relationships.

For example:

- 23:55 and 00:00 are only five minutes apart, even though their raw
  clock values lie near opposite ends of the daily scale;
- Sunday and Monday are adjacent in the weekly cycle;
- December 31 and January 1 are adjacent in the annual cycle.

Periodic calendar quantities are therefore represented using paired
sine and cosine features.

For a periodic variable with phase `x` and period `T`:

\[
\sin\left(2\pi\frac{x}{T}\right),
\qquad
\cos\left(2\pi\frac{x}{T}\right)
\]

are used together.

The following cycles are created:

- daily cycle — 1440 minutes;
- weekly cycle — 7 days;
- annual cycle — 365 or 366 days depending on the year.

Raw calendar components are retained for interpretation, but the
cyclical representations will be preferred as default model features.

In [21]:
# ============================================================
# CELL 21 — CREATE CYCLICAL GREGORIAN FEATURES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Calendar variables such as clock time, weekday, and annual
# position are periodic.
#
# Using raw integer values can create artificial boundaries.
#
# Examples:
#
#     23:55 and 00:00
#     Sunday and Monday
#     December 31 and January 1
#
# should be treated as close rather than far apart.
#
# We therefore create paired sine/cosine representations for:
#
#     - time of day
#     - day of week
#     - annual position
#
# The original calendar columns remain unchanged.


# ============================================================
# 1. DAILY CYCLE
# ============================================================
#
# There are 1440 minutes in one day.
#
# Using minute_of_day gives a precise 5-minute position
# within the daily cycle.

daily_phase = (
    2
    * np.pi
    * df_work["utc_minute_of_day"]
    / 1440
)

df_work["utc_time_sin"] = np.sin(
    daily_phase
)

df_work["utc_time_cos"] = np.cos(
    daily_phase
)


# ============================================================
# 2. WEEKLY CYCLE
# ============================================================
#
# pandas encodes:
#
#     Monday = 0
#     ...
#     Sunday = 6
#
# Sine/cosine encoding ensures Sunday and Monday remain
# adjacent on the circular representation.

weekly_phase = (
    2
    * np.pi
    * df_work["utc_day_of_week"]
    / 7
)

df_work["utc_weekday_sin"] = np.sin(
    weekly_phase
)

df_work["utc_weekday_cos"] = np.cos(
    weekly_phase
)


# ============================================================
# 3. ANNUAL CYCLE
# ============================================================
#
# We make the calculation leap-year aware even though the
# present SOLETE period does not include February 29.
#
# A fractional day component is included so that annual
# position changes smoothly during the day rather than
# jumping only at midnight.

days_in_year = np.where(
    df_work.index.is_leap_year,
    366.0,
    365.0,
)


annual_position_days = (
    (df_work["utc_day_of_year"] - 1)
    +
    (
        df_work["utc_minute_of_day"]
        / 1440.0
    )
)


annual_phase = (
    2
    * np.pi
    * annual_position_days
    / days_in_year
)


df_work["utc_year_sin"] = np.sin(
    annual_phase
)

df_work["utc_year_cos"] = np.cos(
    annual_phase
)


# ============================================================
# 4. VERIFY FEATURE RANGES
# ============================================================

cyclical_calendar_columns = [
    "utc_time_sin",
    "utc_time_cos",
    "utc_weekday_sin",
    "utc_weekday_cos",
    "utc_year_sin",
    "utc_year_cos",
]


print("Cyclical Gregorian feature ranges:\n")

for column in cyclical_calendar_columns:

    print(
        f"{column:20s}: "
        f"{df_work[column].min(): .6f} "
        f"to "
        f"{df_work[column].max(): .6f}"
    )


# ============================================================
# 5. VERIFY CIRCULAR NORMS
# ============================================================
#
# For each sine/cosine pair:
#
#     sin² + cos² = 1
#
# apart from tiny floating-point error.

daily_norm = (
    df_work["utc_time_sin"] ** 2
    +
    df_work["utc_time_cos"] ** 2
)

weekly_norm = (
    df_work["utc_weekday_sin"] ** 2
    +
    df_work["utc_weekday_cos"] ** 2
)

annual_norm = (
    df_work["utc_year_sin"] ** 2
    +
    df_work["utc_year_cos"] ** 2
)


print("\nMaximum daily-cycle norm error:")
print(
    np.abs(daily_norm - 1).max()
)

print("\nMaximum weekly-cycle norm error:")
print(
    np.abs(weekly_norm - 1).max()
)

print("\nMaximum annual-cycle norm error:")
print(
    np.abs(annual_norm - 1).max()
)


# ============================================================
# 6. INSPECT IMPORTANT DAILY BOUNDARY
# ============================================================
#
# Show values around midnight to verify that 23:55 and 00:00
# are close in the circular representation.

midnight_boundary_example = df_work.loc[
    "2018-06-01 23:50:00+00:00":
    "2018-06-02 00:10:00+00:00",
    [
        "utc_minute_of_day",
        "utc_time_sin",
        "utc_time_cos",
    ]
]


print("\nDaily-cycle representation around midnight:")

midnight_boundary_example

Cyclical Gregorian feature ranges:

utc_time_sin        : -1.000000 to  1.000000
utc_time_cos        : -1.000000 to  1.000000
utc_weekday_sin     : -0.974928 to  0.974928
utc_weekday_cos     : -0.900969 to  1.000000
utc_year_sin        : -1.000000 to  1.000000
utc_year_cos        : -1.000000 to  1.000000

Maximum daily-cycle norm error:
2.220446049250313e-16

Maximum weekly-cycle norm error:
2.220446049250313e-16

Maximum annual-cycle norm error:
2.220446049250313e-16

Daily-cycle representation around midnight:


,utc_minute_of_day,utc_time_sin,utc_time_cos
Timestamp,,,
2018-06-01 23:50:00+00:00,1430,-0.043619,0.999048
2018-06-01 23:55:00+00:00,1435,-0.021815,0.999762
2018-06-02 00:00:00+00:00,0,0.000000,1.000000
2018-06-02 00:05:00+00:00,5,0.021815,0.999762
2018-06-02 00:10:00+00:00,10,0.043619,0.999048


## Research Literature Basis

The calendar-based feature-engineering approach in this project is
motivated by three complementary strands of prior research.

### 1. Base Study — Lunar Calendar and Forecasting

Darmawan et al. (2023), *Improving the Forecasting Accuracy Based on the
Lunar Calendar in Modeling Rainfall Levels Using the Bi-LSTM Method
through the Grid Search Approach*, compared Gregorian-calendar and
lunar-calendar rainfall forecasting using a Bi-LSTM model.

The study provides the closest methodological precedent for the present
research because it explicitly evaluates whether an alternative
calendar representation can improve time-series forecasting.

However, the present project differs in several important ways:

- renewable-energy generation is forecast instead of monthly rainfall;
- the SOLETE dataset provides high-resolution 5-minute observations;
- calendar information will initially be introduced as features rather
  than changing the temporal observations being compared;
- identical timestamps, targets, train/test periods, and model
  conditions will be retained across feature-set comparisons;
- Gregorian, astronomical, and Vedic/lunisolar information will be
  evaluated incrementally through controlled ablation experiments.

### 2. Indian Lunisolar Calendar and Weather Patterns

Rana, Mistry, and Raghuwanshi (2014), *Novel Usage of Gujarati Calendar
in Temporal Association Rule Mining for Temperature Analysis of Surat,
India*, investigated temperature patterns using the Vikram Samvat
Gujarati calendar and temporal association-rule mining.

This work is relevant because it provides prior evidence for applying an
Indian lunisolar calendar representation to an environmental variable.

Its objective, however, was association discovery rather than
renewable-energy forecasting.

### 3. Astronomical Computation of Panchang Elements

Anbalagan et al. (2026), *Dedicated Astronomical Computation Unit for
Luni-Solar Calendrical Systems in 90nm CMOS*, demonstrates algorithmic
computation of Hindu astronomical-calendar/Panchang quantities,
including Tithi, Nakshatra, Yoga, and Karana.

This work is relevant to the computational definition and implementation
of Vedic/lunisolar features.

It is not used as evidence that these features improve forecasting;
rather, it supports their treatment as deterministically computable
astronomical-calendar quantities.

### Working Research Direction

Together, these studies motivate investigating whether deterministic
Vedic/lunisolar astronomical information provides predictive information
beyond conventional Gregorian calendar features for renewable-energy
forecasting.

The planned experimental hierarchy is:

1. historical measurements / autoregressive baseline;
2. + Gregorian cyclical features;
3. + independently computed astronomical features;
4. + Vedic/lunisolar calendar features;
5. optional hybrid/decomposition extensions only after the contribution
   of each feature family has been established.

Any claim that Vedic/lunisolar features improve forecasting will be made
only if supported by controlled out-of-sample experiments.

In [22]:
# ============================================================
# CELL 22 — DEFINE EXPERIMENTAL FEATURE FAMILIES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Before creating astronomical and Vedic/lunisolar features,
# we explicitly define the feature families that will later
# be compared experimentally.
#
# This is important because any improvement should be
# attributable to a specific additional source of information.
#
# We do NOT want to create one large feature set containing
# Gregorian, astronomical, and Vedic variables and then
# claim that the Vedic calendar improved forecasting.
#
# Instead, features will be added incrementally.
#
# This cell is documentation only.
# It does NOT modify df_work.


feature_family_plan = pd.DataFrame(
    [
        # ----------------------------------------------------
        # F0 — HISTORICAL / AUTOREGRESSIVE BASELINE
        # ----------------------------------------------------
        {
            "stage": "F0",
            "feature_family": "Historical baseline",

            "examples": (
                "Lagged target power; lagged temperature; "
                "cleaned humidity; wind speed; circular wind "
                "direction; cleaned pressure; historical GHI/POA"
            ),

            "known_future": "No",

            "purpose": (
                "Establish forecasting performance using only "
                "information available from historical observations."
            ),
        },


        # ----------------------------------------------------
        # F1 — GREGORIAN CALENDAR FEATURES
        # ----------------------------------------------------
        {
            "stage": "F1",
            "feature_family": "Gregorian calendar",

            "examples": (
                "utc_time_sin/cos; utc_weekday_sin/cos; "
                "utc_year_sin/cos; optional weekend indicator"
            ),

            "known_future": "Yes",

            "purpose": (
                "Measure the value of conventional deterministic "
                "calendar/time information."
            ),
        },


        # ----------------------------------------------------
        # F2 — SOLAR ASTRONOMICAL FEATURES
        # ----------------------------------------------------
        {
            "stage": "F2",
            "feature_family": "Solar astronomy",

            "examples": (
                "Independently computed solar elevation, solar "
                "azimuth, solar zenith / daylight information"
            ),

            "known_future": "Yes",

            "purpose": (
                "Measure the contribution of physically meaningful "
                "solar-position information beyond Gregorian time."
            ),
        },


        # ----------------------------------------------------
        # F3 — LUNAR ASTRONOMICAL FEATURES
        # ----------------------------------------------------
        {
            "stage": "F3",
            "feature_family": "Lunar astronomy",

            "examples": (
                "Lunar phase / Sun-Moon angular separation and "
                "other independently computed lunar quantities"
            ),

            "known_future": "Yes",

            "purpose": (
                "Separate ordinary lunar astronomical information "
                "from specifically Vedic/lunisolar calendar features."
            ),
        },


        # ----------------------------------------------------
        # F4 — VEDIC / PANCHANG FEATURES
        # ----------------------------------------------------
        {
            "stage": "F4",
            "feature_family": "Vedic / lunisolar calendar",

            "examples": (
                "Tithi; Paksha; Nakshatra; Yoga; Karana; "
                "other carefully verified Panchang quantities"
            ),

            "known_future": "Yes",

            "purpose": (
                "Test whether the Vedic/lunisolar representation "
                "adds predictive information beyond Gregorian and "
                "astronomical variables."
            ),
        },
    ]
)


print("Planned feature-family hierarchy:\n")

feature_family_plan

Planned feature-family hierarchy:



,stage,feature_family,examples,known_future,purpose
0,F0,Historical baseline,Lagged target power; lagged temperature; cleaned humidity; wind speed; circular wind direction; cleaned pressure; historical GHI/POA,No,Establish forecasting performance using only information available from historical observations.
1,F1,Gregorian calendar,utc_time_sin/cos; utc_weekday_sin/cos; utc_year_sin/cos; optional weekend indicator,Yes,Measure the value of conventional deterministic calendar/time information.
2,F2,Solar astronomy,"Independently computed solar elevation, solar azimuth, solar zenith / daylight information",Yes,Measure the contribution of physically meaningful solar-position information beyond Gregorian time.
3,F3,Lunar astronomy,Lunar phase / Sun-Moon angular separation and other independently computed lunar quantities,Yes,Separate ordinary lunar astronomical information from specifically Vedic/lunisolar calendar features.
4,F4,Vedic / lunisolar calendar,Tithi; Paksha; Nakshatra; Yoga; Karana; other carefully verified Panchang quantities,Yes,Test whether the Vedic/lunisolar representation adds predictive information beyond Gregorian and astronomical variables.


## Refined Research Question and Experimental Hypotheses

The original project proposal considered converting a Gregorian time
series into a Vedic/lunisolar calendar representation and combining the
result with decomposition and machine-learning forecasting.

Following dataset exploration and literature review, the research design
has been refined to provide a more controlled comparison.

The underlying 5-minute SOLETE observations will remain on the same
continuous UTC timeline. Alternative calendar systems will initially be
introduced as deterministic feature representations rather than by
changing or re-aggregating the observations themselves.

### Primary Research Question

**Can deterministic Vedic/lunisolar calendar features improve
short-term renewable-energy forecasting beyond historical measurements,
conventional Gregorian time features, and standard astronomical
features?**

### Secondary Research Question

**If Vedic/lunisolar features improve forecasting, do they provide
information beyond the underlying solar and lunar astronomical
quantities from which the calendar is derived?**

### Experimental Hypotheses

- **H0:** Vedic/lunisolar calendar features do not provide a meaningful
  out-of-sample forecasting improvement beyond historical, Gregorian,
  and astronomical feature sets.

- **H1:** Vedic/lunisolar calendar features provide a meaningful
  out-of-sample forecasting improvement beyond historical, Gregorian,
  and astronomical feature sets.

### Controlled Feature-Ablation Strategy

Forecasting experiments will incrementally compare:

- F0 — historical/autoregressive information;
- F1 — F0 + Gregorian calendar features;
- F2 — F1 + solar astronomical features;
- F3 — F2 + lunar astronomical features;
- F4 — F3 + Vedic/Panchang features.

The same target timestamps, chronological train/validation/test periods,
forecast horizons, and comparable model configurations will be retained
across these experiments.

Only after the predictive contribution of these feature families has
been established will signal-decomposition approaches such as VMD and
more complex hybrid forecasting architectures be evaluated.

This design separates the effect of calendar representation from the
effects of signal decomposition and model complexity.

## Final Preprocessing and Gregorian Feature-Engineering Summary

This notebook established the preprocessing and baseline temporal-feature
pipeline for the selected 5-minute SOLETE dataset.

### Data Preservation

The original 11 SOLETE variables were retained unchanged throughout the
analysis. All preprocessing transformations were stored in separate
columns so that the source measurements remain available for
traceability and verification.

### Final Preprocessing Decisions

**Pressure**

- 456 systematic pressure anomalies were identified at 00:00 UTC.
- The anomalies behave as approximately +1000 mbar daily-boundary offsets.
- `Pressure_clean[mbar]` corrects only these confirmed observations by
  subtracting 1000 mbar.
- `pressure_anomaly_flag` records the affected timestamps.

**Humidity**

- 164 humidity observations above 1 were identified.
- All occurred exactly at 00:00 UTC.
- Diagnostic comparison strongly favoured causal forward filling from
  the immediately preceding valid observation.
- `Humidity_clean` retains the resulting 0–1 observed scale without
  imposing an unverified unit conversion.
- `humidity_anomaly_flag` records the affected timestamps.

**Wind Direction**

- 103 source wind-direction observations required modulo-360 wrapping.
- The original direction remains unchanged.
- Wind direction is represented for modelling using:
  - `wind_dir_sin`
  - `wind_dir_cos`
- Raw/wrapped angles and the wrapping flag are retained for diagnostics.

**Gaia Wind-Power Target Quality**

- `P_Gaia[kW]` was not numerically corrected.
- Cross-resolution comparison against independently reconstructed
  1-minute-to-5-minute values identified 48 questionable midnight target
  observations.
- `gaia_target_quality_flag` is retained as an offline dataset-quality
  mask for later wind-target training and evaluation.
- It is not intended as a forecasting input.

### Leakage-Control Principle

Measured meteorological and power variables occurring after the forecast
origin will not be supplied directly to baseline forecasting models.

Historical measurements may be used through lagged or sequential
features, while deterministic future quantities such as calendar and
astronomical variables may be calculated for the forecast timestamp.

### Gregorian Baseline Features

UTC Gregorian components were extracted and the major periodic
quantities were represented cyclically using sine/cosine pairs:

- time of day;
- day of week;
- annual position.

These form the conventional calendar baseline against which subsequent
astronomical and Vedic/lunisolar feature families will be evaluated.

### Experimental Feature Hierarchy

The planned controlled comparison is:

- F0 — historical/autoregressive baseline;
- F1 — F0 + Gregorian features;
- F2 — F1 + solar astronomical features;
- F3 — F2 + lunar astronomical features;
- F4 — F3 + Vedic/Panchang features.

Signal decomposition and more complex hybrid forecasting models will be
investigated only after the incremental contribution of these feature
families has been established.

### Next Stage

The next notebook will independently compute and validate astronomical
features for the DTU/SYSLAB site, beginning with precise site,
timezone, and astronomical-coordinate assumptions.